#**Moelagem Preditiva | Previsão de Demanda de Produtos e Crescimento de Centros de Distribuição**
<img src="/Workspace/Users/cauabrgal11@alu.ufc.br/Drafts/img/LOGO LH NAUTICAL ORIGINAL.png" width="300" alt="Texto alternativo"> <img src="/Workspace/Users/cauabrgal11@alu.ufc.br/Drafts/img/4. IndiciumAI_Logo_Mono_White.png" width="300" alt="Texto alternativo">


#Estudo de Caso 1 - Previsão de Demanda de Produtos

## Glossário
- **SMA**: Simple Moving Average = Média Móvel Simples
- **Lag:** um ponto específico no passado ("exatamente o que aconteceu há N meses"). Exemplo: "Quanto um produto vendeu a 1 mês atrás?".
- **Rolling mean (média móvel)**: a média dos últimos N meses, sem incluir o mês atual.


###Instalação de dependências:

In [0]:
%pip install xgboost
%pip install statsmodels

In [0]:
# Isso reinicia o interpretador Python do notebook pra "enxergar" a lib recém-instalada.
dbutils.library.restartPython()

## Etapa 1 — Reconhecimento dos dados Gold para Previsão de Demanda

Antes de montar qualquer modelo de previsão, é preciso analisar os dados que serão utilizados com base nos seguintes parâmetros:

1. **Volumetria**: quantas linhas existem em `fct_sales_items`. Ela esclarece a escala do problema (quantas combinações produto × loja × dia serão preditas) e serve de referência para detectar perdas de dados em joins futuros.

2. **Cobertura temporal (min/max de `date_sk`)**: período coberto pelos dados. Deve ser um histórico contínuo e suficientemente longo para capturar sazonalidade anual. Além disso é preciso verificar a continuidade da série temporal e a existência de gaps, visto que os modelos de regressão a serem utilizados assumem uma linha do tempo regular (um valor por dia, mesmo que seja zero).

3. **Granularidade de `location_type`**: Foi solicitada a previsão "por loja" e, 
   separadamente, "crescimento por centro de distribuição". No atual modelo dimensional, ambos são `location_id`s com `location_type` diferentes (`store` vs `warehouse`). É preciso confirmar que a coluna promove a distinção dos dois tipos de forma limpa antes de usá-la como critério de agrupamento nas perguntas de negócio.

OBS: Essa etapa é puramente exploratória, nada está sendo transformado ainda.

##Resultados da Etapa 1 — Reconhecimento dos dados

- **Volumetria**: 150.321 linhas em `fct_sales_items`.
- **Período coberto**: `2020-01-01` a `2026-12-31`
- **6 locais no total**: 3 lojas (`store`) e 3 armazéns 
  (`warehouse`).
- Foi confirmado que os armazéns também vendem diretamente 
  (não são apenas logística) e possuem volume de venda parecido entre `store` e `warehouse`.

In [0]:
from datetime import date
from pyspark.sql import functions as F

# Cell 1 - Verificação das tabelas gold a serem utilizadas
# AS_OF_DATE_SK: último dia do último mês fechado.
# Data ancorada em 31/08/2026 para garantir que nenhum mês parcial seja incluído na análise.
AS_OF_DATE_SK = 20260831

df_sales = (
    spark.table("workspace.gold.fct_sales_items")
    .filter(F.col("date_sk") <= AS_OF_DATE_SK)
)
df_dates = spark.table("workspace.gold.dim_dates")
df_loc   = spark.table("workspace.gold.dim_location")
df_prod  = spark.table("workspace.gold.dim_products")

print("fct_sales_items (filtrado até o último mês fechado):", df_sales.count(), "linhas")
# Define se será possível treinar modelo por produto×loja individualmente ou se será necessário agregar em grupos maiores

df_sales.select("date_sk").agg({"date_sk": "min"}).show()
df_sales.select("date_sk").agg({"date_sk": "max"}).show()
# Calcula a quantidade de meses/dias de histórico existentes e define se é possível realizar previsões com 3 meses de confiança e isolar a sazonalidade anual


display(df_sales.limit(5))
display(df_loc.groupBy("location_type").count())

# Quantidade de valores distintos de location_type existentes e número de linhas por tipo — especialmente warehouses
## Caso existam apenas 1-2 warehouses, a análise do "crescimento por centro de distribuição" terá pouca base de comparação, alterando a abordagem do item sobre sugestão de novos centros de distribuição

###Resultados:
* Existem 3 lojas e 3 armazéns (warehouse), o que possibilita a previsão por local individualmente sem a necessidade de agrupamento de lojas por volume para reduzir dimensionalidade. 3 é gerenciável.

## Etapa 2 — Granularidade e completude da série temporal

### Justificativa metodológica

A previsão de demanda requer uma estrutura temporal regular. Um registro por 
período (dia, semana ou mês) para cada combinação produto × loja, mesmo quando não 
houve transação naquele intervalo. Nesses casos, o valor deve ser representado 
como zero, não como ausência de registro. A ausência de uma grade completa 
introduz viés na estimação, visto que séries com poucos pontos podem apresentar 
variabilidade artificial, e features baseadas em lags ou médias móveis tornam-se 
inconsistentes.

Antes da definição da granularidade temporal (diária, semanal ou mensal), duas 
questões devem ser avaliadas:

1. **Dimensionalidade do problema**: O número de combinações produto × loja 
   determina a viabilidade de modelos individuais por série. Quando o volume é 
   reduzido (poucas dezenas), é possível ajustar modelos específicos por série. 
   Quando o volume é elevado (milhares), modelos globais que aprendem padrões 
   de múltiplas séries simultaneamente, utilizando produto e loja como features 
   categóricas são mais apropriados.

2. **Densidade temporal dos dados**: Bases de dados sintéticas ou com padrões de 
   crescimento não uniformes podem apresentar volume crescente ao longo do tempo 
   (e.g., poucos registros em 2020 e volume significativamente maior a partir de 
   2024). Essa característica impacta diretamente a escolha do baseline e a confiabilidade do histórico disponível para treinamento.

Considerando que o horizonte de previsão requerido é de **3 meses**, e que a 
aplicação de negócio envolve decisões de compra e distribuição (não previsão de 
picos diários), adota-se a **granularidade mensal por produto × loja** como grão 
principal de modelagem. Essa granularidade é suficiente para capturar padrões de 
sazonalidade e tendência, além de apresentar maior robustez a ruído em comparação 
com a granularidade diária. A granularidade diária permanece disponível para 
análises de sazonalidade específicas, caso necessário.

## Resultados da Etapa 2 — Granularidade e completude

- **1.009 produtos × 6 locais = 6.054 combinações**, revelando que **todo produto já vendeu 
  em todo local** ao menos uma vez. Grade "cheia" em termos de existência.
- **Tendência de crescimento clara**: vendas subindo de 76 mil (2020) pra 
  174 mil (2026) unidades/ano, ~12-15% ao ano. 
- Volume de venda bem distribuído entre os 6 locais (nenhum muito destoante).

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

# Cálculo da dimensionalidade do problema: combinações produto × loja
n_variants = df_sales.select("variant_sk").distinct().count()
n_locations = df_sales.select("location_sk").distinct().count()
n_combos = df_sales.select("variant_sk", "location_sk").distinct().count()
## Quando n_combos é próximo de n_variants × n_locations, indica que todos os produtos foram vendidos em todos os locais
## Esta métrica orienta a escolha entre modelagem por série individual ou modelo global, e identifica séries com histórico insuficiente


print(f"Produtos distintos (variant_sk): {n_variants}")
print(f"Locais distintos (location_sk): {n_locations}")
print(f"Combinações produto x loja: {n_combos}")

# Densidade dos dados ao longo do tempo: registros por ano
df_sales_yearly = (
    df_sales
    .withColumn("year", F.substring(F.col("date_sk").cast("string"), 1, 4))
    .groupBy("year")
    .agg(F.count("*").alias("num_registros"), F.sum("quantity").alias("qtd_total"))
    .orderBy("year")
)
display(df_sales_yearly)
## Verifica se o volume apresenta crescimento suave ou saltos artificiais (e.g., 2020-2022 com baixo volume seguido de crescimento acentuado em 2024+)
## Caso o início do histórico seja excessivamente esparso, esses períodos devem ser excluídos do treino para evitar que ruído comprometa o baseline de médias móveis


# Confirma o cruzamento location_sk <-> location_type
df_sales_loc = (
    df_sales
    .drop("location_id")
    .join(df_loc, on="location_sk", how="left")
)
## Ambas as tabelas fct_sales_items e dim_location possuem uma coluna location_id. O location_id da tabela fato é removido antes do join para evitar ambiguidade, mantendo apenas o location_id de dim_location.


display(
    df_sales_loc
    .groupBy("location_id", "location_name", "location_type")
    .agg(F.countDistinct("variant_sk").alias("produtos_distintos"), F.sum("quantity").alias("qtd_total"))
    .orderBy("location_type", "location_id")
)
## Confirma nominalmente os 3 stores e 3 warehouses e verifica se a quantidade total de cada um é comparável ou apresenta desbalanceamento significativo
## Volume substancialmente inferior em um CD pode indicar subutilização ou lacuna de cobertura.


# Gráfico
pdf_sales_yearly = df_sales_yearly.orderBy("year").toPandas()

plt.figure(figsize=(8, 4))
plt.plot(pdf_sales_yearly["year"], pdf_sales_yearly["qtd_total"], marker="o")
plt.title("Quantidade vendida por ano (2020–2026)")
plt.xlabel("Ano")
plt.ylabel("Quantidade total vendida")
plt.tight_layout()
plt.show()

###Insights Extraídos - Etapa 2

1. Observa-se uma tendência de crescimento clara e suave, ano a ano, com taxa aproximada de 12-15% ao ano, resultando em crescimento de 76 mil para 174 mil unidades.

2. A malha de combinações produto×loja apresenta cobertura completa (1009 × 6 = 6054 combinações), indicando que todo produto foi vendido em todo local ao menos uma vez. Contudo, a densidade temporal é limitada: 150.321 registros de venda ÷ 6054 combinações produto×loja ≈ 25 transações por combinação ao longo de 7 anos (~84 meses), resultando em média inferior a 1 venda a cada 3 meses por combinação produto-loja.

Essa métrica evidencia demanda intermitente estrutural no catálogo. A maioria dos produtos apresenta padrão de venda esporádica, o que impacta diretamente a estratégia de modelagem:

A previsão mês a mês, produto a produto, loja a loja, tende a gerar séries com elevada frequência de valores nulos ou próximos a zero. Modelos baseados em médias móveis simples tendem a prever valores próximos a zero e apresentar métricas de erro favoráveis (MAE baixo), porém com baixa utilidade para decisões de compra.

Antes da definição da estratégia de modelagem (séries individuais versus agregação por categoria ou segmentação por volume de giro), faz-se necessária a medição formal da intermitência, além da estimativa por média.

## Etapa 3 — Medindo a intermitência da demanda (antes da agregação)

Nesta etapa será aplicada uma taxonomia clássica de padrões de demanda: a Syntetos-Boylan. Ela categoriza os produtos em quatro classes com base em duas métricas:

**CV²** (Coeficiente de Variação ao Quadrado): (desvio padrão / média)² calculado exclusivamente sobre os períodos com demanda positiva. Esta métrica quantifica a variabilidade da demanda quando esta ocorre, valores elevados indicam alta volatilidade nas quantidades vendidas entre períodos consecutivos.

**ADI** (Average Demand Interval): razão entre o número total de períodos do calendário e o número de períodos com vendas positivas. Esta métrica quantifica a intermitência do produto — valores próximos de 1 indicam vendas consistentes em praticamente todos os períodos; valores superiores denotam intervalos prolongados sem transações.

**Limiares clássicos**: CV² = 0,49 e ADI = 1,32 definem quatro categorias:

- **Smooth** (ADI < 1,32 e CV² < 0,49): demanda regular e estável.
- **Erratic** (ADI < 1,32 e CV² ≥ 0,49): vendas frequentes com alta variabilidade nas quantidades.
- **Intermittent** (ADI ≥ 1,32 e CV² < 0,49): baixa frequência de vendas, porém com quantidades estáveis quando ocorrem.
- **Lumpy** (ADI ≥ 1,32 e CV² ≥ 0,49): padrão de maior complexidade — baixa frequência de vendas com alta variabilidade nas quantidades; a previsibilidade é naturalmente limitada.

A classificação é relevante porque a seleção do método de previsão mais adequado depende do padrão de demanda observado: produtos com padrão *smooth* apresentam bom desempenho com modelos simples, enquanto produtos *lumpy* requerem técnicas especializadas ou podem apresentar erro elevado independentemente do método escolhido.

In [0]:
def classificar_demanda(serie_mensal, meses_totais):
    """
    serie_mensal: array com a demanda mensal (incluindo meses com 0, se houver)
    meses_totais: quantidade de períodos no calendário completo (pra achar os "buracos")
    """
    vendas_positivas = serie_mensal[serie_mensal > 0]

    if len(vendas_positivas) < 2:
        return np.nan, np.nan, "dados_insuficientes"

    cv2 = (vendas_positivas.std() / vendas_positivas.mean()) ** 2
    adi = meses_totais / len(vendas_positivas)

    if adi < 1.32 and cv2 < 0.49:
        categoria = "smooth"
    elif adi < 1.32 and cv2 >= 0.49:
        categoria = "erratic"
    elif adi >= 1.32 and cv2 < 0.49:
        categoria = "intermittent"
    else:
        categoria = "lumpy"

    return cv2, adi, categoria


In [0]:
from pyspark.sql import functions as F
import pandas as pd

sdf = (
    df_sales
    .filter(F.col("invoice_status").isNotNull())
    .join(df_prod.select("variant_sk", "product_name"), on="variant_sk", how="left")
    .withColumn("year_month", F.substring(F.col("date_sk").cast("string"), 1, 6))
    .groupBy("year_month", "variant_sk", "product_name")
    .agg(F.sum("net_quantity").alias("net_quantity"))
    .orderBy("year_month")
)

df = sdf.toPandas()
df["month_start"] = pd.to_datetime(df["year_month"], format="%Y%m")

# Classificação Syntetos-Boylan (ADI x CV²) — mesma função do Cell 15
todos_meses = pd.date_range(df["month_start"].min(), df["month_start"].max(), freq="MS")

resumo_produtos = []
for pv_id, nome in df.groupby(["variant_sk", "product_name"]).groups.keys():
    serie_prod = (
        df[df["variant_sk"] == pv_id]
        .groupby("month_start")["net_quantity"]
        .sum()
        .reindex(todos_meses, fill_value=0)
    )
    cv2, adi, categoria = classificar_demanda(serie_prod.values, len(todos_meses))
    resumo_produtos.append({
        "variant_sk": pv_id,
        "product_name": nome,
        "total_vendido": serie_prod.sum(),
        "meses_com_venda": (serie_prod > 0).sum(),
        "cv2": cv2,
        "adi": adi,
        "categoria": categoria,
    })

classificacao = pd.DataFrame(resumo_produtos).sort_values("total_vendido", ascending=False)
classificacao["categoria"].value_counts()

## Resultados da Etapa 3 — Classificação Syntetos-Boylan (net_quantity, pedidos faturados)

Ao classificar os 1.009 produtos do catálogo da LH Nautical com base na demanda
líquida mensal (net_quantity, excluindo pedidos não faturados), foi obtido:

| Categoria | Produtos | % do catálogo |
| --- | --- | --- |
| Intermittent | 553 | 54,8% |
| Lumpy | 444 | 44,0% |
| Erratic | 8 | 0,8% |
| Smooth | 4 | 0,4% |

**Quase 99% do catálogo (997 de 1.009 produtos) apresenta demanda intermitente
ou lumpy**, apenas 12 produtos (~1,2%) têm padrão de venda regular (smooth) ou
errático. Isso revela três coisas importantes para o negócio:

1. **A intermitência é estrutural, não pontual.** Não trata-se de alguns produtos
   de baixo giro que "puxam" a média para baixo; o catálogo como um todo vende de
   forma esparsa. Mesmo produtos com bom volume total concentram vendas em poucos
   meses, com longos intervalos sem nenhuma transação. Isso é consistente com o
   perfil da LH Nautical: demanda sazonal, movida por eventos específicos
   (início de temporada, promoções, reposição de estoque), e não por consumo
   contínuo.

2. **44% dos produtos são lumpy**, o cenário mais desafiável para previsão.
   Produtos lumpy combinam baixa frequência de vendas (ADI elevado) com alta
   variabilidade nas quantidades quando a venda ocorre (CV² elevado). Para esses
   itens, métodos de previsão baseados em médias móveis ou regressão linear simples
   tendem a produzir estimativas próximas de zero que, embora possam apresentar
   métricas de erro absoluto favoráveis (MAE baixo), não geram valor para decisões
   de compra e reposição.

**Conclusão de negócio:** a LH Nautical não pode depender de previsões série-a-série
para planejamento de compra e distribuição. A intermitência generalizada exige um
modelo que aprenda do catálogo inteiro e que, para produtos lumpy, possa oferecer
estimativas probabilísticas (intervalos de confiança) em vez de previsões
pontuais, já que o valor esperado individual é baixo, mas o risco de ruptura ou
excesso de estoque permanece alto.

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

# sales_date_range: descobre o período REAL de vendas (min e max de date_sk em fct_sales_items,
sales_date_range = df_sales.select(
    F.min("date_sk").alias("min_date_sk"),
    F.max("date_sk").alias("max_date_sk"),
).collect()[0]
print("Range real de vendas:", sales_date_range["min_date_sk"], "a", sales_date_range["max_date_sk"])

# df_calendar_months: lista de meses, restrita ao período REAL de vendas
# Grão: 1 linha = 1 mês. Colunas: year_month (formato yyyyMM)
df_calendar_months = (
    df_dates
    .filter(
        (F.col("date_sk") >= sales_date_range["min_date_sk"])
        & (F.col("date_sk") <= sales_date_range["max_date_sk"])
    )
    .withColumn("year_month", F.substring(F.col("date_sk").cast("string"), 1, 6))
    .select("year_month")
    .distinct()
)
print("Total de meses distintos no calendário (filtrado):", df_calendar_months.count())

# Agregação mensal por produto x loja 
df_monthly = (
    df_sales
    .withColumn("year_month", F.substring(F.col("date_sk").cast("string"), 1, 6))
    .groupBy("variant_sk", "location_sk", "year_month")
    .agg(F.sum("quantity").alias("qty_month"))
)
## Grão: 1 linha = 1 combinação (variant_sk, location_sk, year_month) que teve pelo menos 1 venda no mês. 
## Meses sem venda NÃO aparecem aqui 
### Colunas: variant_sk, location_sk, year_month (formato yyyyMM), qty_month (soma vendida no mês)


# Para cada combinação produto x loja: em quantos dos meses reais houve venda?
total_months = df_calendar_months.count()  # calculado dinamicamente a partir do range real de vendas

# df_intermittency: um resumo por combinação produto x loja de quão intermitente é a demanda.
df_intermittency = (
    df_monthly
    .groupBy("variant_sk", "location_sk")
    .agg(
        F.count("year_month").alias("meses_com_venda"),
        F.sum("qty_month").alias("qty_total_combo"),
        F.avg("qty_month").alias("media_qty_quando_vende"),
    )
    .withColumn("pct_meses_com_venda", F.round(F.col("meses_com_venda") / F.lit(total_months) * 100, 1))
)
## Grão: 1 linha = 1 combinação (variant_sk, location_sk) — resume os meses possíveis em um único registro.
## Colunas:
##   meses_com_venda      -> em quantos meses essa combinação teve pelo menos 1 venda
##   qty_total_combo      -> soma de tudo que essa combinação vendeu no período inteiro
##   media_qty_quando_vende -> média de unidades vendidas, considerando só os meses em que vendeu (exclui zeros)
##   pct_meses_com_venda  -> meses_com_venda / total_months, em % (nosso indicador principal de intermitência)


# Visão geral: distribuição estatística do % de meses ativos entre todas as combinações
display(df_intermittency.select("pct_meses_com_venda").summary("min", "25%", "50%", "75%", "max"))
## A mediana e o intervalo (min→max): estão concentrados numa faixa estreita (ex: todos entre 10%-40%) ou bem espalhados (alguns quase 0%, outros quase 100%)?
## Faixa estreita = intermitência estrutural do catálogo inteiro (aponta pra modelo global). Faixa espalhada = existe segmento de alto giro e cauda longa (aponta pra segmentar a estratégia)


# Top 15 produtos por volume total
df_top15 = (
    df_intermittency
    .groupBy("variant_sk")
    .agg(
        F.sum("qty_total_combo").alias("qty_total_produto"),
        F.avg("pct_meses_com_venda").alias("pct_medio_meses_ativos"),
    )
    .join(
        df_prod.select("variant_sk", "product_name", "variant_name"),
        on="variant_sk",
        how="left",
    )
    .withColumn("produto_label", F.concat_ws(" - ", F.col("product_name"), F.col("variant_name")))
    .orderBy(F.desc("qty_total_produto"))
    .limit(15)
)
display(df_top15)
## Olha a coluna pct_medio_meses_ativos desses campeões de venda: está bem acima da mediana geral, ou parecida?
## Se até os mais vendidos tiverem % parecido com o resto do catálogo, confirma que não dá pra usar "volume vendido" como critério de segmentação — reforça a decisão de modelo global
### df_top15: os 15 produtos (agregando todas as lojas juntas) que mais venderam no período,
### Grão: 1 linha = 1 produto (variant_sk), volume somado de todas as 6 lojas.
### Colunas:
###   qty_total_produto      -> soma de tudo que esse produto vendeu, em todas as lojas, no período inteiro
###   pct_medio_meses_ativos -> média do pct_meses_com_venda desse produto entre as lojas onde ele é vendido
###   product_name / variant_name / produto_label -> identificação legível vinda de dim_products


# --- Gráfico 1: distribuição de % de meses ativos ---
pdf_intermittency = df_intermittency.select("pct_meses_com_venda").toPandas()
plt.figure(figsize=(8, 4))
plt.hist(pdf_intermittency["pct_meses_com_venda"], bins=20, color="#2E86AB", edgecolor="#1A4D6D", alpha=0.85)
plt.title("Distribuição do % de meses com venda por combinação produto x loja", fontsize=12, fontweight="bold")
plt.xlabel("% de meses ativos", fontsize=10)
plt.ylabel("Número de combinações produto x loja", fontsize=10)
plt.grid(axis="y", alpha=0.3, linestyle="--")
plt.tight_layout()
plt.show()
## Gráfico 1: histograma do df_intermittency -> mostra COMO os % de meses ativos se distribuem entre as combinações (pico único = catálogo homogêneo; bimodal = dois grupos)
### Formato da distribuição: um pico único e estreito, ou dois "morros" (bimodal)? Um pico único confirma estrutura homogênea. Bimodal indicaria dois grupos de comportamento distintos que vale segmentar


# --- Gráfico 2: top 15 produtos por volume total ---
pdf_top15 = df_top15.toPandas()
plt.figure(figsize=(10, 6))
bars = plt.barh(pdf_top15["produto_label"], pdf_top15["qty_total_produto"], color="#F77F00", edgecolor="#A35200", alpha=0.85)
plt.gca().invert_yaxis()
plt.title("Top 15 produtos por volume total vendido", fontsize=12, fontweight="bold")
plt.xlabel("Quantidade total vendida", fontsize=10)
plt.grid(axis="x", alpha=0.3, linestyle="--")
plt.tight_layout()
plt.show()
## As barras têm tamanhos parecidos entre si, ou tem 1-2 produtos muito destacados dos demais?
## Se for parecido, reforça que não existe um pequeno grupo de "campeões absolutos" dominando o catálogo — todos os produtos têm participação relativamente distribuída
## Gráfico 2: barra do df_top15 -> mostra os 15 produtos campeões de venda, já com nome legível

### Análise dos Resultados - Etapa 3

A distribuição do percentual de meses ativos apresenta padrão **unimodal**, com pico único concentrado na faixa de **22-24%**, sem evidência de bimodalidade. A maior parte das combinações produto×loja situa-se entre 20% e 28% de meses ativos, com uma cauda que se estende até aproximadamente 38-40%. Essa distribuição homogênea indica **ausência de segmentos claramente distintos** entre produtos de alta e baixa frequência de venda — não existe uma divisão natural "alto giro vs. cauda longa" que justifique estratégias separadas.

Entre os 15 produtos de maior volume total vendido, observa-se:

* **Volumes relativamente uniformes**: os totais variam de aproximadamente **900 a 1.030 unidades**, sem concentração extrema em poucos itens dominantes. Não há um ou dois "campeões absolutos" que se destacam drasticamente dos demais — a distribuição de volume é equilibrada mesmo no topo do ranking.

* **Intermitência consistente**: embora sejam os produtos de maior giro, seu percentual de meses ativos permanece próximo à mediana geral do catálogo (~23-25%). Isso reforça que **até os mais vendidos apresentam padrão de demanda esparsa**, confirmando que o volume total vendido não é um bom critério para segmentar estratégias de previsão.

* **Lacuna de qualidade de dados**: um dos 15 produtos de maior volume aparece como **"Produto Não Identificado" (LHN-595227)**, indicando problema de cadastro ou integração que afeta inclusive o segmento de maior movimentação. Essa lacuna de identificação pode limitar a capacidade de interpretação de negócio dos resultados do modelo e deve ser endereçada em etapa posterior de governança de dados.

**Implicação metodológica:** Os resultados observados sustentam a adoção de um **modelo de regressão global** (single model, não segmentado por volume ou frequência de giro), utilizando produto e loja como features categóricas. Esta decisão está fundamentada em três evidências: (1) distribuição unimodal e homogênea da intermitência, (2) ausência de "campeões absolutos" que justificassem tratamento especial, e (3) padrão de demanda esparsa que persiste até nos produtos de maior giro. A abordagem global permite que o modelo "empreste" padrão entre séries parecidas, maximizando a aprendizagem em um contexto de dados escassos por série individual.

## Etapa 4 — Construção da grade calendário completa (zero-fill)

### Justificativa: requisito para features de lag e média móvel

Uma lacuna numa série de demanda possui significado de negócio preciso: 
**zero vendas naquele período**, não ausência de informação. Entretanto, o `df_monthly` da 
Etapa 3, ao agregar via `groupBy`, gera uma linha apenas quando existe pelo 
menos uma venda no período, meses sem venda não produzem registro algum.

Ao calcular-se a "quantidade vendida no mês anterior" (`lag`) diretamente sobre 
essa tabela incompleta, o "mês anterior" corresponderia ao último mês em 
que ocorreu venda, o qual pode situar-se 2, 3 ou 4 meses atrás — não ao mês calendário 
imediatamente anterior. Tal distorção contamina quaisquer features de histórico recente 
com informação incorreta.

A solução consiste em gerar o **produto cartesiano** de todos os `variant_sk` × todos 
os `location_sk` × todos os 84 meses do período, executando um `left join` com 
o `df_monthly` real e preenchendo com `0` onde não houver correspondência.

## Insights Extraídos - Etapa 4

- Foi constatado que `dim_dates` cobre um range bem mais longo que o 
  histórico real de vendas. Sem o filtro, seriam considerados "meses fantasmas" que não existiam no negócio, o que ocasionaria distorção em qualquer métrica de intermitência posterior. Essa é a base "oficial" de dados usada daqui pra frente — toda feature de 
  série temporal parte dela.

In [0]:
from pyspark.sql import functions as F

# NOTA: sales_date_range e df_calendar_months foram calculados na Etapa 3
# (cálculo de intermitência requer essas variáveis), portanto não há necessidade de recálculo.
# Reutilizam-se as variáveis já existentes.

# df_all_combos: todas as combinações produto x loja observadas nos dados de venda,
# independentemente da frequência de ocorrência. Grão: 1 linha = 1 combinação (variant_sk, location_sk).
df_all_combos = df_sales.select("variant_sk", "location_sk").distinct()
print("Total de combinações produto x loja:", df_all_combos.count())

# df_grid: produto cartesiano combos x meses = a "grade" completa e regular que será utilizada
# como base de qualquer feature de série temporal.
# Grão: 1 linha = 1 combinação (variant_sk, location_sk, year_month), sempre presente,
# mesmo quando não há venda registrada naquele mês.
df_grid = df_all_combos.crossJoin(df_calendar_months)
print("Tamanho esperado da grade completa (combos x meses):", df_grid.count())
## Resultado esperado: 6054 x 80 = 484.320 (não mais 508.536 — alteração decorrente da redução de 84 para 80 meses)

# df_monthly_full: grade completa com vendas reais onde registradas e 0 onde ausentes.
# Grão: igual ao df_grid. Colunas: variant_sk, location_sk, year_month, qty_month (0 quando não há venda)
df_monthly_full = (
    df_grid
    .join(df_monthly, on=["variant_sk", "location_sk", "year_month"], how="left")
    .fillna(0, subset=["qty_month"])
)
print("Linhas na grade final (deve corresponder ao valor esperado acima):", df_monthly_full.count())
## Os dois números (grade esperada vs linhas finais) devem ser idênticos.
## Divergência indica duplicação ou perda de linhas no join — possível chave duplicada.

# Teste de sanidade: recálculo de pct_meses_com_venda sobre a grade completa
# deve produzir EXATAMENTE os mesmos números da Etapa 3 (min 7.5, mediana 23.8, max 41.3)
display(
    df_monthly_full
    .groupBy("variant_sk", "location_sk")
    .agg(F.avg((F.col("qty_month") > 0).cast("int")).alias("pct_meses_com_venda_check"))
    .select("pct_meses_com_venda_check")
    .summary("min", "25%", "50%", "75%", "max")
)
## Deve corresponder exatamente à Etapa 3 (min 7.5 / 25% → 21.3 / mediana 23.8 / 75% → 27.5 / max 41.3)
## Teste de regressão: correspondência exata valida a correção da grade zero-fill.

## Etapa 5 — Engenharia de features para o modelo de regressão

### Justificativa para a seleção de features

Um modelo de regressão não possui, por si só, a capacidade de reconhecer 
estrutura temporal nos dados , é necessário transformar padrões de calendário 
e histórico recente em colunas explícitas. Serão construídos 4 grupos de features:

1. **Lags (valores de meses anteriores)**: `lag_1`, `lag_2`, `lag_3` e `lag_12`.
   Os três primeiros capturam a inércia recente da demanda (o comportamento 
   observado há 1-3 meses tende a se repetir). O `lag_12` é particularmente 
   relevante para a sazonalidade: o volume vendido exatamente 12 meses atrás 
   permite ao modelo aprender picos recorrentes anuais (exemplo: alta de itens 
   náuticos no verão).

2. **Médias móveis (rolling mean)**: média dos últimos 3 e 6 meses. Estas 
   correspondem ao **baseline** (médias móveis) estabelecido, agora incorporadas como *features* do modelo de regressão — permitindo 
   ao modelo aprender quando confiar mais na média recente versus quando 
   o padrão apresenta mudança estrutural.

3. **Calendário**: número do mês (1-12) e trimestre. Essas features permitem 
   ao modelo aprender sazonalidade mesmo em séries com histórico limitado, 
   uma vez que o padrão de cada mês pode ser inferido a partir de outras 
   combinações produto×loja semelhantes (conforme a decisão da Etapa 3: 
   modelo global, não segmentado por série).

4. **Tendência**: um índice sequencial de mês (1, 2, 3... até 84), para 
   capturar o crescimento ano a ano observado na Etapa 2 (~13% ao ano). 
   Sem essa feature, o modelo não seria capaz de incorporar a informação 
   de que o negócio como um todo apresenta crescimento contínuo.

### Consideração importante: valores nulos nos primeiros meses de cada série

Uma combinação produto×loja possui "mês anterior" disponível apenas a partir 
do 2º mês da grade, e "há 12 meses" apenas a partir do 13º mês. Consequentemente, 
os primeiros 12 meses de cada uma das 6.054 séries **não terão** todas as 
features preenchidas — trata-se de comportamento esperado, não erro. Será 
identificado quantas linhas são afetadas, para posterior decisão sobre 
descarte dessas observações no treinamento ou 
preenchimento com valor padrão.

## Insights Extraídos — Etapa 5

- Foram construídas 4 famílias de features: **lags** (1, 2, 3 e 12 meses anteriores), 
  **médias móveis** (janelas de 3 e 6 meses), **calendário** (mês e trimestre) e 
  **tendência** (índice sequencial de mês, para capturar o crescimento 
  ano a ano observado na Etapa 2).
- Foi realizada validação manual em uma série real (Colete Salva-Vidas) para 
  confirmar o alinhamento temporal correto dos lags e médias móveis, sem 
  vazamento de informação futura para o passado, o que invalidaria qualquer 
  modelo treinado sobre esses dados.
- **Consideração técnica**: os primeiros 12 meses de cada uma das 6.054 séries 
  permanecem sem `lag_12` preenchido (72.648 linhas no total). Este comportamento 
  é esperado e será tratado no processo de separação treino/teste, uma vez que 
  essas linhas iniciais não dispõem de histórico completo para treinamento.
- Foi implementada uma otimização de performance: o índice de mês é calculado 
  em uma tabela separada (84 linhas) e incorporado via `broadcast join`, em vez 
  de ordenar as 508 mil linhas completas em uma única partição. 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# Window: define a "janela" de cada série (partição por produto x loja, ordenada no tempo).
# year_month no formato yyyyMM ordena corretamente como string, já que todos têm o mesmo tamanho.
w = Window.partitionBy("variant_sk", "location_sk").orderBy("year_month")

# df_month_index: tabela pequena (84 linhas) que mapeia cada year_month a um índice sequencial.
# Calculado separado do df_features pra evitar ordenar as 508k linhas inteiras numa única
# partição (isso gerava o warning "No Partition Defined for Window operation").
# Grão: 1 linha = 1 mês. Colunas: year_month, month_index (1..84)
df_month_index = (
    df_monthly_full
    .select("year_month")
    .distinct()
    .withColumn("month_index", F.dense_rank().over(Window.orderBy("year_month")))
)

# df_features: df_monthly_full + as 4 famílias de features (lags, rolling mean, calendário, tendência).
# Grão: igual ao df_monthly_full -> 1 linha = 1 combinação (variant_sk, location_sk, year_month).
# Colunas novas:
#   lag_1, lag_2, lag_3, lag_12  -> qty_month de N meses atrás (mesma combinação)
#   roll_mean_3m, roll_mean_6m  -> média móvel dos últimos 3 / 6 meses (SEM incluir o mês atual)
#   month_num, quarter          -> extraídos de year_month, capturam sazonalidade de calendário
#   month_index                 -> índice sequencial global (1..84), captura tendência/crescimento
#                                  (vem de um join com df_month_index, não mais calculado direto aqui)
df_features = (
    df_monthly_full
    .withColumn("lag_1", F.lag("qty_month", 1).over(w))
    .withColumn("lag_2", F.lag("qty_month", 2).over(w))
    .withColumn("lag_3", F.lag("qty_month", 3).over(w))
    .withColumn("lag_12", F.lag("qty_month", 12).over(w))
    .withColumn("roll_mean_3m", F.avg("qty_month").over(w.rowsBetween(-3, -1)))
    .withColumn("roll_mean_6m", F.avg("qty_month").over(w.rowsBetween(-6, -1)))
    .withColumn("month_num", F.col("year_month").substr(5, 2).cast("int"))
    .withColumn("quarter", F.ceil(F.col("month_num") / 3))
    .join(F.broadcast(df_month_index), on="year_month", how="left")  # tabela pequena -> broadcast join, evita shuffle
)

# Checagem: quantas linhas ficaram com pelo menos uma feature de lag nula
# (esperado: só os primeiros meses de cada série, já que ainda não tem histórico suficiente)
total_rows = df_features.count()
rows_with_null_lag12 = df_features.filter(F.col("lag_12").isNull()).count()

print(f"Total de linhas na grade: {total_rows}")
## Total de linhas na grade	Deve ser 508536 (igual à Etapa 4)	Confirma que não perdemos/duplicamos nada ao adicionar colunas

print(f"Linhas sem lag_12 preenchido (primeiros 12 meses de cada série): {rows_with_null_lag12}")
print(f"Esperado teórico: 6054 combinações x 12 meses = {6054 * 12}")
## Linhas sem lag_12 vs Esperado teórico (72648)	Os dois números devem bater	Se não bater, algo na window function está errado (ex: partição incompleta)

# Preview: inspeciona visualmente uma série específica (o produto mais vendido, na primeira loja)
# pra confirmar que lag/rolling estão alinhados corretamente com o calendário
display(
    df_features
    .filter(F.col("variant_sk") == "430c3626b879b4005d41b8a46172e0c0")  # Colete Salva-Vidas 2374, top 1 do ranking
    .orderBy("location_sk", "year_month")
    .select("location_sk", "year_month", "qty_month", "lag_1", "lag_2", "lag_3", "lag_12", "roll_mean_3m", "roll_mean_6m", "month_num", "quarter", "month_index")
    .limit(20)
)
## Preview da série do "Colete Salva-Vidas"	Olha as primeiras linhas (jan/2020): lag_1, lag_2, lag_3, lag_12 devem estar null. 
## A partir da 13ª linha (jan/2021), lag_12 deve mostrar o valor de qty_month de exatamente 12 linhas acima. 
## roll_mean_3m na linha N deve ser a média de qty_month das 3 linhas anteriores (não incluindo a linha N)	

##OBS: Decisão de escopo
Como setembro de 2026 está em andamento, os dados válidos para uso serão até o último mês fechado (agosto/2026).


#Resumo Metodológico — Etapas 1 a 6

## 1. Definição do Baseline

A Etapa 6 estabeleceu o modelo de referência para avaliação de desempenho preditivo: médias móveis simples de 3 e 6 meses (SMA-3 e SMA-6). Este baseline não constitui um modelo de aprendizado de máquina no sentido estrito, trata-se de uma heurística determinística que utiliza exclusivamente o histórico recente de cada série temporal. A escolha deste método como referência justifica-se por três razões técnicas:

1. **Simplicidade interpretável**: permite estabelecer um piso mínimo de desempenho sem custos computacionais significativos.
2. **Ausência de overfitting**: não há risco de ajuste excessivo aos dados históricos, uma vez que não existe fase de treinamento.
3. **Ampla utilização na indústria**: médias móveis representam a abordagem padrão para previsão de demanda em ambientes de negócio sem infraestrutura de ML.

O modelo de regressão supervisionada (Etapa 7) será avaliado em relação a este baseline, permitindo quantificar o ganho incremental obtido pela incorporação de técnicas de aprendizado de máquina.

## 2. Arquitetura do Modelo: Abordagem Global vs. Segmentada

A decisão de arquitetura mais relevante para este projeto, estabelecida na Etapa 3 e implementada nas etapas subsequentes, refere-se ao escopo de modelagem:

- **Baseline (Etapa 6)**: por definição, cada média móvel é calculada individualmente para cada uma das 6.054 combinações produto×loja. Embora tecnicamente sejam 6.054 heurísticas independentes, as métricas de desempenho são agregadas para avaliação conjunta.

- **Modelo de Regressão (Etapa 7)**: foi adotada uma **arquitetura global única**, na qual um único modelo aprende simultaneamente os padrões das 6.054 séries temporais. Os identificadores de produto (`variant_sk`) e loja (`location_sk`) são incorporados como variáveis preditoras (features), juntamente com lags temporais, indicadores de calendário e tendência.

### Justificativa Técnica da Abordagem Global

Esta decisão fundamenta-se nos seguintes princípios:

1. **Compartilhamento de padrões entre séries**: produtos de categorias semelhantes ou lojas em regiões similares apresentam comportamentos correlacionados. Um modelo global pode explorar estas regularidades transversais.

2. **Aprendizado eficaz em séries com histórico limitado**: combinações produto×loja recém-introduzidas ou com vendas esparsas beneficiam-se do conhecimento extraído de séries mais estabelecidas.

3. **Escalabilidade operacional**: a manutenção e atualização de um único modelo é significativamente mais eficiente do que o gerenciamento de milhares de modelos independentes.

4. **Captura de interações não-lineares**: o modelo global pode identificar regras complexas do tipo "produto P na loja L no mês M tende a apresentar demanda Y", sem necessidade de engenharia manual de interações.

## 3. Estratégia de Validação: Testes em Múltiplos Cenários Sazonais

A metodologia de avaliação, aplicável tanto ao baseline quanto ao modelo de regressão, emprega **validação temporal com múltiplos horizontes de teste**:

- **Cenário 1 (Inverno/Baixa Temporada)**: treinamento até maio/2026, teste em junho/julho/agosto 2026.
- **Cenário 2 (Verão/Alta Temporada)**: treinamento até novembro/2025, teste em dezembro/2025, janeiro/fevereiro 2026.

### Esclarecimento Metodológico

Esta estratégia **não implica o treinamento de modelos distintos para cada estação**. Um único modelo (ou, no caso do baseline, uma única parametrização) é treinado e, em seguida, testado contra dois períodos temporais com características sazonais distintas. Os objetivos desta abordagem são:

1. **Avaliar robustez sazonal**: quantificar se o desempenho do modelo degrada significativamente em períodos de alta vs. baixa demanda.
2. **Identificar vieses sistemáticos**: detectar se o modelo sub ou superestima de forma consistente em contextos específicos.
3. **Garantir comparabilidade**: ao submeter baseline e modelo de regressão aos mesmos cenários de teste, assegura-se que a comparação de desempenho seja justa e não confundida por diferenças nos períodos de avaliação.

Esta técnica de validação cruzada temporal constitui uma prática recomendada em previsão de séries temporais, mitigando o risco de que o desempenho medido seja resultado de particularidades de um único período de teste.

In [0]:
from pyspark.sql import functions as F

# cutoff_month_index: treina até maio/2026 (month_index 77), testa contra
# jun/jul/ago 2026 (78, 79, 80) -> os 3 últimos meses que REALMENTE aconteceram.
cutoff_month_index = 77

# df_train / df_test: split temporal da grade de features (não aleatório).
df_train = df_features.filter(F.col("month_index") <= cutoff_month_index)
df_test = df_features.filter(F.col("month_index") > cutoff_month_index)

print("Linhas de treino:", df_train.count())
print("Linhas de teste:", df_test.count())
## Linhas de treino / Linhas de teste -- Teste deve ter 6054 × 3 = 18.162 -- Confirma que pegou exatamente 3 meses completos

print("Meses de teste:", [r["year_month"] for r in df_test.select("year_month").distinct().orderBy("year_month").collect()])
## Meses de teste	Deve mostrar ['202606', '202607', '202608']	Confirma que são jun/jul/ago 2026 — meses que realmente aconteceram, não mais dado sintético "futuro"


# % de linhas de teste com demanda real = 0 -> justifica usar WAPE além de MAE/RMSE/MAPE
pct_zero_actual = df_test.filter(F.col("qty_month") == 0).count() / df_test.count() * 100
print(f"% de linhas de teste com demanda real = 0: {pct_zero_actual:.1f}%")
## % de demanda real = 0	Deve continuar numa faixa alta (60-75%), coerente com a Etapa 3	Justifica o uso de WAPE no relatório

# df_baseline_snapshot: "congela" a média móvel de cada combinação no primeiro mês de teste
# (month_index = 78). Nesse ponto, roll_mean_3m/6m usam só meses de TREINO (75-77 e 72-77),
# então não há vazamento de dado de teste.
# Grão: 1 linha por combinação (variant_sk, location_sk).
df_baseline_snapshot = (
    df_features
    .filter(F.col("month_index") == cutoff_month_index + 1)
    .select(
        "variant_sk", "location_sk",
        F.col("roll_mean_3m").alias("baseline_sma3"),
        F.col("roll_mean_6m").alias("baseline_sma6"),
    )
)

# df_test_with_baseline: cada linha real de teste (jun/jul/ago) recebe a MESMA previsão
# congelada acima -> é assim que um baseline de média móvel "prevê" múltiplos meses à frente.
df_test_with_baseline = (
    df_test
    .select("variant_sk", "location_sk", "year_month", "month_index", F.col("qty_month").alias("actual"))
    .join(df_baseline_snapshot, on=["variant_sk", "location_sk"], how="left")
)

# df_baseline_errors: erro de cada previsão baseline vs. o valor real
df_baseline_errors = (
    df_test_with_baseline
    .withColumn("error_sma3", F.col("actual") - F.col("baseline_sma3"))
    .withColumn("error_sma6", F.col("actual") - F.col("baseline_sma6"))
)

# Métricas agregadas: MAE, RMSE e WAPE para as duas variantes do baseline
metrics = df_baseline_errors.agg(
    F.round(F.avg(F.abs("error_sma3")), 3).alias("mae_sma3"),
    F.round(F.sqrt(F.avg(F.pow("error_sma3", 2))), 3).alias("rmse_sma3"),
    F.round(F.sum(F.abs("error_sma3")) / F.sum("actual"), 4).alias("wape_sma3"),
    F.round(F.avg(F.abs("error_sma6")), 3).alias("mae_sma6"),
    F.round(F.sqrt(F.avg(F.pow("error_sma6", 2))), 3).alias("rmse_sma6"),
    F.round(F.sum(F.abs("error_sma6")) / F.sum("actual"), 4).alias("wape_sma6"),
)
display(metrics)
## Tabela de métricas	SMA-3 ou SMA-6 tem menor erro?	Define qual variante do baseline vira "a" referência oficial pra comparar com o modelo de regressão



# ============================================================
# COMPLEMENTO: segundo backtest, agora na ALTA TEMPORADA (verão: dez/jan/fev)
# Motivo: o teste acima (jun/jul/ago) é baixa temporada (inverno) -- valida só
# metade do cenário de negócio. 
# ============================================================

# cutoff_year_month_peak: mês de corte do treino pro segundo backtest (treina até aqui)
cutoff_year_month_peak = "202511"
test_months_peak = ["202512", "202601", "202602"]

cutoff_month_index_peak = (
    df_month_index.filter(F.col("year_month") == cutoff_year_month_peak)
    .collect()[0]["month_index"]
)

# df_test_peak: só os 3 meses de pico de verão (dez/jan/fev), pra teste
df_test_peak = df_features.filter(F.col("year_month").isin(test_months_peak))
print("Linhas de teste (pico de verão):", df_test_peak.count())
## Deve ser 6054 x 3 = 18.162, igual ao backtest de inverno

pct_zero_actual_peak = df_test_peak.filter(F.col("qty_month") == 0).count() / df_test_peak.count() * 100
print(f"% de linhas de teste com demanda real = 0 (alta temporada): {pct_zero_actual_peak:.1f}%")
## Deve ser MENOR que os 82.0% do teste de inverno -> evidência numérica direta da sazonalidade de verão

# df_baseline_snapshot_peak: mesma lógica do baseline de inverno, mas "congelado" no mês
# seguinte ao corte de nov/2025 -> usa só meses de TREINO (set/out/nov 2025), sem vazamento
df_baseline_snapshot_peak = (
    df_features
    .filter(F.col("month_index") == cutoff_month_index_peak + 1)
    .select(
        "variant_sk", "location_sk",
        F.col("roll_mean_3m").alias("baseline_sma3"),
        F.col("roll_mean_6m").alias("baseline_sma6"),
    )
)

# df_test_with_baseline_peak: cada linha real de teste (dez/jan/fev) recebe a MESMA previsão
# congelada acima -> mesma lógica de "baseline repetido" do backtest de inverno
df_test_with_baseline_peak = (
    df_test_peak
    .select("variant_sk", "location_sk", "year_month", F.col("qty_month").alias("actual"))
    .join(df_baseline_snapshot_peak, on=["variant_sk", "location_sk"], how="left")
)

# df_baseline_errors_peak: erro de cada previsão baseline vs. o valor real, no pico de verão
df_baseline_errors_peak = (
    df_test_with_baseline_peak
    .withColumn("error_sma3", F.col("actual") - F.col("baseline_sma3"))
    .withColumn("error_sma6", F.col("actual") - F.col("baseline_sma6"))
)

# Métricas agregadas: MAE, RMSE e WAPE para as duas variantes do baseline, no pico de verão
metrics_peak = df_baseline_errors_peak.agg(
    F.round(F.avg(F.abs("error_sma3")), 3).alias("mae_sma3"),
    F.round(F.sqrt(F.avg(F.pow("error_sma3", 2))), 3).alias("rmse_sma3"),
    F.round(F.sum(F.abs("error_sma3")) / F.sum("actual"), 4).alias("wape_sma3"),
    F.round(F.avg(F.abs("error_sma6")), 3).alias("mae_sma6"),
    F.round(F.sqrt(F.avg(F.pow("error_sma6", 2))), 3).alias("rmse_sma6"),
    F.round(F.sum(F.abs("error_sma6")) / F.sum("actual"), 4).alias("wape_sma6"),
)
display(metrics_peak)
## Compara com a tabela "metrics" do teste de inverno: WAPE menor ou maior aqui?
## Qualquer resultado é insight -> confirma (ou não) que médias móveis simples
## conseguem lidar com sazonalidade, e justifica o uso de lag_12 no modelo de regressão

## Etapa 7 — Primeiro modelo de regressão (Random Forest)

### Justificativa Técnica

O algoritmo Random Forest foi selecionado como primeiro modelo de regressão por três razões técnicas:

1. **Capacidade de processar features categóricas de alta cardinalidade**: a variável `variant_sk` apresenta 1.009 valores distintos, e o Random Forest lida adequadamente com essa característica mediante indexação numérica simples, sem exigir técnicas sofisticadas de encoding.
2. **Independência de normalização ou padronização de features numéricas**: diferentemente de modelos lineares, o Random Forest apresenta robustez a diferenças de escala entre variáveis (lags e médias móveis).
3. **Capacidade de modelar relações não-lineares**: o algoritmo captura naturalmente interações complexas entre produto, localização e período temporal, sem necessidade de engenharia manual de features de interação.

Este modelo representa um primeiro estágio de modelagem, acima do baseline, com o objetivo de validar o potencial de algoritmos mais sofisticados (Gradient Boosting, LightGBM) em etapas subsequentes.

### Especificação das features e respectivas justificativas

- **`variant_sk`, `location_sk`** (categóricas, indexadas): identificadores que permitem ao modelo aprender simultaneamente os padrões das 6.054 séries temporais, caracterizando uma abordagem de modelagem global.
- **`lag_1`, `lag_2`, `lag_3`, `lag_12`**: histórico recente e valor do mesmo mês no ano anterior, projetados para capturar padrões sazonais não identificados pelo baseline.
- **`roll_mean_3m`, `roll_mean_6m`**: médias móveis que representam o nível recente de atividade, mesma informação utilizada isoladamente no baseline, aqui incorporada como uma feature adicional.
- **`month_num`, `quarter`**: indicadores de calendário explícitos, destinados a reforçar padrões sazonais mesmo em séries com histórico limitado, mediante transferência de conhecimento de séries com comportamento similar.
- **`month_index`**: índice sequencial que captura tendência de crescimento temporal identificada na Etapa 2.

### Tratamento de valores ausentes

As primeiras 12 observações de cada série temporal não apresentam valores válidos para `lag_12` (valores `null`). Essas observações serão excluídas do conjunto de treinamento e teste, em conformidade com os requisitos de modelos de regressão supervisionada, que não aceitam valores ausentes como entrada.

### Metodologia de validação: equivalência com baseline para comparação controlada

O modelo será treinado e testado nos mesmos dois cenários temporais utilizados na avaliação do baseline:
1. Treinamento até maio/2026 → teste em junho/julho/agosto 2026 (período de baixa demanda sazonal)
2. Treinamento até novembro/2025 → teste em dezembro/2025, janeiro/fevereiro 2026 (período de alta demanda sazonal)

Essa estratégia não implica o desenvolvimento de modelos distintos por região ou temporada. Trata-se do mesmo algoritmo e conjunto de features, executado em dois cortes temporais distintos, com o objetivo de avaliar o desempenho em condições sazonais diferenciadas. Essa abordagem permite comparação direta e controlada com as métricas do baseline previamente estabelecidas.

In [0]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

# Colunas de feature: categóricas (serão indexadas) + numéricas (usadas direto)
CATEGORICAL_COLS = ["variant_sk", "location_sk"]
NUMERIC_COLS = ["lag_1", "lag_2", "lag_3", "lag_12", "roll_mean_3m", "roll_mean_6m", "month_num", "quarter", "month_index"]
LABEL_COL = "qty_month"

# df_features_clean: df_features sem as linhas dos primeiros 12 meses de cada série
# (onde lag_12 é null) -- modelo de regressão não aceita null como entrada.
df_features_clean = df_features.dropna(subset=NUMERIC_COLS)
print("Linhas antes de remover nulls:", df_features.count())
print("Linhas depois de remover nulls:", df_features_clean.count())


def treinar_e_avaliar(df_features_clean, cutoff_month_index=None, cutoff_year_month=None, test_months=None, label=""):
    """
    Treina 1 Random Forest com tudo até o corte informado, testa nos meses informados,
    e retorna as métricas (MAE, RMSE, WAPE) no mesmo formato usado no baseline (Etapa 6).
    Usa cutoff_month_index (número) OU cutoff_year_month (string) -- o backtest de inverno
    usa índice numérico, o de verão usa string, exatamente como fizemos na Etapa 6.
    """
    if cutoff_month_index is not None:
        df_train = df_features_clean.filter(F.col("month_index") <= cutoff_month_index)
    else:
        cutoff_idx = (
            df_month_index.filter(F.col("year_month") == cutoff_year_month).collect()[0]["month_index"]
        )
        df_train = df_features_clean.filter(F.col("month_index") <= cutoff_idx)

    df_test = df_features_clean.filter(F.col("year_month").isin(test_months))

    # Pipeline: indexa as categóricas, monta o vetor de features, treina o Random Forest.
    # maxBins precisa ser >= número de categorias distintas de variant_sk (1.009), senão o Spark quebra.
    indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in CATEGORICAL_COLS]
    assembler = VectorAssembler(
        inputCols=[f"{c}_idx" for c in CATEGORICAL_COLS] + NUMERIC_COLS,
        outputCol="features",
    )
    rf = RandomForestRegressor(
        featuresCol="features", labelCol=LABEL_COL,
        numTrees=50, maxDepth=8, maxBins=1100, seed=42,
    )
    pipeline = Pipeline(stages=indexers + [assembler, rf])

    model = pipeline.fit(df_train)
    df_pred = model.transform(df_test)

    # Métricas no mesmo formato do baseline (Etapa 6), pra comparação direta
    df_errors = df_pred.withColumn("error", F.col(LABEL_COL) - F.col("prediction"))
    metrics = df_errors.agg(
        F.round(F.avg(F.abs("error")), 3).alias("mae"),
        F.round(F.sqrt(F.avg(F.pow("error", 2))), 3).alias("rmse"),
        F.round(F.sum(F.abs("error")) / F.sum(LABEL_COL), 4).alias("wape"),
    ).withColumn("cenario", F.lit(label))

    return model, df_pred, metrics


# Rodada 1: mesmo cenário do baseline de inverno (treino até mai/2026, teste jun/jul/ago 2026)
model_winter, df_pred_winter, metrics_winter = treinar_e_avaliar(
    df_features_clean, cutoff_month_index=77, test_months=["202606", "202607", "202608"], label="inverno"
)

# Rodada 2: mesmo cenário do baseline de verão (treino até nov/2025, teste dez/2025-fev/2026)
model_summer, df_pred_summer, metrics_summer = treinar_e_avaliar(
    df_features_clean, cutoff_year_month="202511", test_months=["202512", "202601", "202602"], label="verão"
)

print("=== Random Forest - Inverno (jun/jul/ago 2026) ===")
display(metrics_winter)

print("=== Random Forest - Verão (dez/jan/fev 2026) ===")
display(metrics_summer)

# Importância das features no modelo de inverno -- ajuda a confirmar (ou não) se lag_12
# realmente "pesou" na decisão do modelo, como esperávamos na teoria
importances = list(zip(CATEGORICAL_COLS + NUMERIC_COLS, model_winter.stages[-1].featureImportances.toArray()))
importances_sorted = sorted(importances, key=lambda x: -x[1])
for feature, imp in importances_sorted:
    print(f"{feature}: {imp:.4f}")


##Insights extraídos - Etapa 7

A avaliação revelou que o modelo de regressão não apresentou ganho preditivo em relação ao baseline de médias móveis simples (SMA-3 e SMA-6). As métricas de erro agregado (MAE, RMSE e WAPE) permaneceram numericamente equivalentes ou, em alguns casos, superiores às do baseline, indicando que a incorporação de técnicas de aprendizado de máquina não gerou valor incremental mensurável neste estágio.

Este resultado não configura falha metodológica, mas evidência de que a engenharia de features atual não captura suficientemente os padrões de demanda, ou que o algoritmo selecionado não é adequado para a estrutura do problema. 

## Etapa 8 — Análise de escala entre produtos e incorporação de features de categoria

### Justificativa para análise de escala neste estágio

A estratégia de modelagem global, um único modelo treinado para todas as 
séries temporais, apresenta uma implicação metodológica relevante: produtos 
com volumes de venda substancialmente superiores aos demais contribuem de forma 
desproporcionalmente elevada para as métricas de erro absoluto (MAE/RMSE) 
agregadas, podendo dominar o processo de otimização do modelo. Essa dinâmica 
resulta em ajuste inadequado para produtos de menor volume, sem que tal 
deficiência seja claramente evidenciada nas métricas agregadas.

### Justificativa para incorporação de features de categoria

A análise de sazonalidade apresentada no dashboard da LH Nautical demonstra que o padrão 
temporal de sazonalidade é compartilhado entre categorias, não sendo 
necessário o desenvolvimento de modelos separados por categoria, porém o 
nível médio de demanda apresenta variação significativa entre elas (a categoria 
Hélices, por exemplo, apresenta volume de vendas superior às demais). A 
incorporação de `category_name` e `parent_category_name` como features permite 
ao modelo capturar essa informação de nível de forma direta, sendo 
particularmente relevante para produtos com histórico limitado, que podem 
beneficiar-se do padrão de nível observado na categoria à qual pertencem.

In [0]:
from pyspark.sql import functions as F

# df_qty_scale: distribuição de qty_month (não-zero) por produto, pra checar
# se há diferença grande de escala entre produtos que pode estar distorcendo
# o erro agregado do modelo.
# Grão: 1 linha = 1 produto (variant_sk).
df_qty_scale = (
    df_monthly_full
    .filter(F.col("qty_month") > 0)  # só meses com venda, pra não distorcer com zeros
    .groupBy("variant_sk")
    .agg(
        F.avg("qty_month").alias("media_qty_quando_vende"),
        F.stddev("qty_month").alias("desvio_qty"),
        F.max("qty_month").alias("max_qty"),
    )
)

display(df_qty_scale.select("media_qty_quando_vende", "desvio_qty", "max_qty").summary("min", "25%", "50%", "75%", "max"))

# df_prod_categories: traz category_name e parent_category_name de dim_products,
# uma linha por variant_sk (mesmo grão de df_prod).
df_prod_categories = df_prod.select("variant_sk", "category_name", "parent_category_name")

# Confere quantas categorias distintas existem, e o volume por categoria
# (mesma lógica do gráfico do dashboard, mas em unidades, não receita)
display(
    df_monthly_full
    .join(df_prod_categories, on="variant_sk", how="left")
    .groupBy("category_name")
    .agg(F.sum("qty_month").alias("qty_total_categoria"))
    .orderBy(F.desc("qty_total_categoria"))
)

# df_features agora recebe category_name e parent_category_name como novas colunas.
# Vamos usar essas colunas como features categóricas extras no próximo treino do modelo.
df_features = df_features.join(df_prod_categories, on="variant_sk", how="left")
print("Categorias distintas nulas após o join:", df_features.filter(F.col("category_name").isNull()).count())

## Etapa 9 — Retreino do modelo com features de categoria

### Alterações metodológicas em relação à Etapa 7

O algoritmo (Random Forest) e a metodologia de teste (dois cenários temporais: 
inverno e verão, idênticos à Etapa 7 e ao baseline da Etapa 6) permanecem 
inalterados. A única modificação consiste na adição de `category_name` e 
`parent_category_name` como novas features categóricas, motivada pela observação 
do gráfico de sazonalidade do dashboard: o formato da sazonalidade é 
compartilhado entre categorias, mas o nível médio de demanda apresenta 
variação significativa (Hélices > Eletrônica > Pesca > ...). A hipótese é que 
essa informação permita ao modelo estimar melhor o nível de demanda de produtos 
com histórico limitado, mediante transferência do padrão observado na categoria.

### Comparação de resultados

Serão comparados três conjuntos de resultados, aplicados aos mesmos dois cenários (inverno/verão):
1. Baseline (médias móveis) — disponível na Etapa 6
2. Random Forest sem features de categoria — disponível na Etapa 7
3. Random Forest com features de categoria — presente nesta Etapa 9

Caso a incorporação de categoria gere valor preditivo mensurável, espera-se 
observar redução do MAE/WAPE em relação à Etapa 7, e importância de feature 
relevante (não nula) atribuída a `category_name` e `parent_category_name` 
(não necessariamente superior a `variant_sk`, mas superior à importância 
observada para `lag_12` na Etapa 7).

## Insights extraídos - Etapa 9

Foram testadas as features `category_name` e `parent_category_name`, motivadas pela observação do gráfico de sazonalidade do dashboard, que evidencia padrão temporal compartilhado entre categorias, com variação apenas no nível médio de demanda.

**Resultado**: não foi observado ganho mensurável nas métricas de erro (MAE/WAPE praticamente idênticos ou levemente superiores em ambos os cenários; importância atribuída à categoria inferior a 1%).

**Justificativa**: dado que cada produto pertence a exatamente uma categoria, essa informação encontra-se implicitamente codificada em `variant_sk`, variável que representa aproximadamente 58-59% da importância das features no modelo. A categoria não acrescenta informação discriminatória adicional no contexto de um catálogo com histórico completo para todos os produtos.

**Decisão metodológica**: as features de categoria não serão mantidas no modelo oficial, preservando simplicidade sem comprometimento de desempenho. A categoria apresentaria relevância em cenários de "cold start" (produtos novos sem histórico), situação não aplicável ao presente conjunto de dados.

Este teste constitui evidência de rigor metodológico para o relatório técnico, demonstrando que as decisões de feature engineering foram fundamentadas em experimentação empírica, não em suposições intuitivas.

In [0]:
from pyspark.sql import functions as F

# Atualiza a lista de colunas categóricas, incluindo as novas de categoria.
# NUMERIC_COLS e LABEL_COL continuam os mesmos da Etapa 7 -- reaproveitados, não redefinidos.
CATEGORICAL_COLS = ["variant_sk", "location_sk", "category_name", "parent_category_name"]

# df_features_clean precisa ser recalculado, porque df_features mudou (ganhou colunas
# de categoria na Etapa 8) desde a última vez que geramos essa versão "sem nulls".
df_features_clean = df_features.dropna(subset=NUMERIC_COLS)
print("Linhas depois de remover nulls (deve bater com a Etapa 7):", df_features_clean.count())

# Reaproveita a MESMA função treinar_e_avaliar definida na Etapa 7 -- não precisa recriar,
# ela já aceita qualquer lista de CATEGORICAL_COLS/NUMERIC_COLS através das variáveis globais.

# Rodada 1: cenário de inverno, agora com categoria como feature
model_winter_v2, df_pred_winter_v2, metrics_winter_v2 = treinar_e_avaliar(
    df_features_clean, cutoff_month_index=77, test_months=["202606", "202607", "202608"], label="inverno_com_categoria"
)

# Rodada 2: cenário de verão, agora com categoria como feature
model_summer_v2, df_pred_summer_v2, metrics_summer_v2 = treinar_e_avaliar(
    df_features_clean, cutoff_year_month="202511", test_months=["202512", "202601", "202602"], label="verao_com_categoria"
)

print("=== Random Forest COM categoria - Inverno ===")
display(metrics_winter_v2)

print("=== Random Forest COM categoria - Verão ===")
display(metrics_summer_v2)

# Comparação lado a lado: modelo sem categoria (Etapa 7) vs. com categoria (Etapa 9)
comparacao = metrics_winter.unionByName(metrics_summer).withColumn("modelo", F.lit("RF_sem_categoria")).unionByName(
    metrics_winter_v2.unionByName(metrics_summer_v2).withColumn("modelo", F.lit("RF_com_categoria"))
)
display(comparacao.orderBy("cenario", "modelo"))

# Importância das features do modelo de inverno com categoria
importances_v2 = list(zip(CATEGORICAL_COLS + NUMERIC_COLS, model_winter_v2.stages[-1].featureImportances.toArray()))
importances_v2_sorted = sorted(importances_v2, key=lambda x: -x[1])
for feature, imp in importances_v2_sorted:
    print(f"{feature}: {imp:.4f}")

## Etapa 10 — Segundo modelo de regressão: XGBoost (Gradient Boosting)

### Justificativa para avaliação de algoritmo alternativo

Random Forest treina árvores **independentes** e calcula a média entre elas. 
XGBoost (Gradient Boosting) treina árvores **em sequência**, onde cada nova 
árvore é otimizada para corrigir os erros residuais das anteriores. Essa 
abordagem tende a capturar padrões não-lineares mais sutis em dados tabulares, 
conforme indicado pela análise exploratória realizada no dashboard da LH Nautical.

A metodologia de teste das Etapas 6/7 (dois cenários temporais: inverno e verão) 
será mantida inalterada, garantindo comparabilidade direta dos resultados.

##Resultados - Etapa 10

### Desempenho comparativo dos modelos

**Cenário inverno (baixa temporada)**: XGBoost apresentou desempenho superior 
ao Random Forest e ao baseline de médias móveis, com redução de 30% no MAE e 
aproximadamente 80 pontos percentuais no WAPE. Este resultado confirma a 
hipótese de que um modelo de boosting sequencial, que corrige iterativamente 
os erros residuais, captura mais adequadamente os padrões não-lineares 
observados na análise exploratória.

**Cenário verão (alta temporada)**: os três modelos apresentaram desempenho 
equivalente (WAPE ~115%). Interpreta-se que, em períodos de demanda regular 
e frequente, mesmo modelos simples capturam adequadamente o nível de vendas. 
O valor incremental de modelos mais sofisticados manifesta-se predominantemente 
em períodos de demanda intermitente ou irregular.

In [0]:
# teste - xgboost ja esta instalado?
try:
    from xgboost.spark import SparkXGBRegressor
    print("✅ xgboost.spark disponível, pode seguir direto.")
except ImportError:
    print("⚠️ Não encontrado. Rode em outra célula: %pip install xgboost, depois reinicie o Python (dbutils.library.restartPython()) e tente de novo.")

In [0]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

# Restaura as colunas oficiais (sem categoria, decisão da Etapa 9)
CATEGORICAL_COLS = ["variant_sk", "location_sk"]
NUMERIC_COLS = ["lag_1", "lag_2", "lag_3", "lag_12", "roll_mean_3m", "roll_mean_6m", "month_num", "quarter", "month_index"]
LABEL_COL = "qty_month"


def treinar_e_avaliar_generico(df_features_clean, estimator, cutoff_month_index=None, cutoff_year_month=None, test_months=None, label=""):
    """
    Versão genérica da função da Etapa 7: recebe qualquer 'estimator' do pyspark.ml
    (RandomForestRegressor, SparkXGBRegressor, GBTRegressor...) já configurado, e roda
    o mesmo pipeline de treino/teste/métricas -- garante comparação 100% justa entre modelos.
    Não substitui a função 'treinar_e_avaliar' da Etapa 7 (que é específica pro Random Forest)
    -- essa aqui é só usada a partir da Etapa 10 em diante.
    """
    if cutoff_month_index is not None:
        df_train = df_features_clean.filter(F.col("month_index") <= cutoff_month_index)
    else:
        cutoff_idx = df_month_index.filter(F.col("year_month") == cutoff_year_month).collect()[0]["month_index"]
        df_train = df_features_clean.filter(F.col("month_index") <= cutoff_idx)

    df_test = df_features_clean.filter(F.col("year_month").isin(test_months))

    indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in CATEGORICAL_COLS]
    assembler = VectorAssembler(inputCols=[f"{c}_idx" for c in CATEGORICAL_COLS] + NUMERIC_COLS, outputCol="features")
    pipeline = Pipeline(stages=indexers + [assembler, estimator])

    model = pipeline.fit(df_train)
    df_pred = model.transform(df_test)

    df_errors = df_pred.withColumn("error", F.col(LABEL_COL) - F.col("prediction"))
    metrics = df_errors.agg(
        F.round(F.avg(F.abs("error")), 3).alias("mae"),
        F.round(F.sqrt(F.avg(F.pow("error", 2))), 3).alias("rmse"),
        F.round(F.sum(F.abs("error")) / F.sum(LABEL_COL), 4).alias("wape"),
    ).withColumn("cenario", F.lit(label))

    return model, df_pred, metrics

In [0]:
import xgboost as xgb_lib
import pandas as pd
from pyspark.sql import functions as F

# Codificação das categóricas: calculada UMA VEZ sobre TODO o df_features_clean,
# não separadamente em treino/teste -- garante que o mesmo produto/loja sempre
# vire o mesmo código numérico, mesmo que apareça só no teste ou só no treino.
pdf_all = df_features_clean.select(*CATEGORICAL_COLS).distinct().toPandas()
cat_mappings = {
    col: {val: i for i, val in enumerate(pdf_all[col].unique())}
    for col in CATEGORICAL_COLS
}


def treinar_e_avaliar_xgb(df_features_clean, cutoff_month_index=None, cutoff_year_month=None, test_months=None, label=""):
    """
    Versão em pandas (não distribuída) do XGBoost, necessária porque o compute
    serverless do Databricks Free Edition bloqueia configs exigidas pela versão
    distribuída (xgboost.spark). Mesmas features, mesmas métricas das outras versões.
    """
    if cutoff_month_index is not None:
        df_train_spark = df_features_clean.filter(F.col("month_index") <= cutoff_month_index)
    else:
        cutoff_idx = df_month_index.filter(F.col("year_month") == cutoff_year_month).collect()[0]["month_index"]
        df_train_spark = df_features_clean.filter(F.col("month_index") <= cutoff_idx)

    df_test_spark = df_features_clean.filter(F.col("year_month").isin(test_months))

    cols_needed = CATEGORICAL_COLS + NUMERIC_COLS + [LABEL_COL]
    pdf_train = df_train_spark.select(*cols_needed).toPandas()
    pdf_test = df_test_spark.select(*cols_needed).toPandas()

    # Aplica a mesma codificação (calculada acima) nos dois conjuntos
    for col in CATEGORICAL_COLS:
        pdf_train[f"{col}_idx"] = pdf_train[col].map(cat_mappings[col])
        pdf_test[f"{col}_idx"] = pdf_test[col].map(cat_mappings[col])

    feature_cols = [f"{c}_idx" for c in CATEGORICAL_COLS] + NUMERIC_COLS

    model = xgb_lib.XGBRegressor(n_estimators=100, max_depth=6, random_state=42)
    model.fit(pdf_train[feature_cols], pdf_train[LABEL_COL])

    pdf_test["prediction"] = model.predict(pdf_test[feature_cols])
    pdf_test["error"] = pdf_test[LABEL_COL] - pdf_test["prediction"]

    mae = pdf_test["error"].abs().mean()
    rmse = (pdf_test["error"] ** 2).mean() ** 0.5
    wape = pdf_test["error"].abs().sum() / pdf_test[LABEL_COL].sum()

    # CORREÇÃO: força conversão explícita pra tipos nativos do Python (float/str),
    # porque numpy.float64 (o tipo padrão do pandas) às vezes não é inferido
    # corretamente pelo spark.createDataFrame -- especialmente em compute serverless.
    metrics_row = {
        "mae": float(round(mae, 3)),
        "rmse": float(round(rmse, 3)),
        "wape": float(round(wape, 4)),
        "cenario": str(label),
    }
    metrics = spark.createDataFrame([metrics_row])

    return model, pdf_test, metrics


model_winter_xgb, pdf_pred_winter_xgb, metrics_winter_xgb = treinar_e_avaliar_xgb(
    df_features_clean, cutoff_month_index=77, test_months=["202606", "202607", "202608"], label="inverno_xgb"
)
model_summer_xgb, pdf_pred_summer_xgb, metrics_summer_xgb = treinar_e_avaliar_xgb(
    df_features_clean, cutoff_year_month="202511", test_months=["202512", "202601", "202602"], label="verao_xgb"
)

print("=== XGBoost - Inverno ===")
display(metrics_winter_xgb)
print("=== XGBoost - Verão ===")
display(metrics_summer_xgb)

# Comparação final: baseline vs. Random Forest (Etapa 7) vs. XGBoost, nos 2 cenários
comparacao_final = (
    metrics_winter.withColumn("modelo", F.lit("RandomForest"))
    .unionByName(metrics_summer.withColumn("modelo", F.lit("RandomForest")))
    .unionByName(metrics_winter_xgb.withColumn("modelo", F.lit("XGBoost")))
    .unionByName(metrics_summer_xgb.withColumn("modelo", F.lit("XGBoost")))
)
display(comparacao_final.orderBy("cenario", "modelo"))

## Comparação preliminar dos três algoritmos testados

| Modelo | MAE (inverno) | WAPE (inverno) | MAE (verão) | WAPE (verão) |
|---|---|---|---|---|
| Baseline (SMA-3) | 2.80 | 260% | 3.81 | 115% |
| Random Forest | 2.14 | 199% | 3.81 | 115% |
| XGBoost | 1.96 | 182% | 3.84 | 116% |

**Cenário inverno (baixa temporada)**: XGBoost apresentou desempenho superior aos demais algoritmos, com redução de 30% no MAE (1.96 vs 2.80) e aproximadamente 80 pontos percentuais no WAPE (182% vs 260%) em relação ao baseline. Este resultado confirma a hipótese de que modelos de boosting sequencial capturam mais adequadamente os padrões não-lineares em comparação com médias móveis simples.

**Cenário verão (alta temporada)**: os três modelos apresentaram desempenho equivalente (WAPE 115-116%, MAE 3.81-3.84). Esta convergência não constitui falha metodológica, mas reflete a natureza mais regular e frequente da demanda no período (55% dos meses com atividade vs 18% no inverno), onde mesmo modelos simples capturam adequadamente o nível de vendas, limitando o valor incremental de abordagens mais sofisticadas.

## Etapa 11 — Walk-forward validation (validação de robustez temporal)

### Justificativa metodológica

As avaliações anteriores (Etapas 7 e 10) testaram o XGBoost em dois 
recortes temporais específicos (jun/jul/ago 2026 e dez/2025-fev/2026). 
Embora informativos, esses testes apresentam limitação: não permitem 
distinguir desempenho consistente de coincidência favorável em períodos 
específicos. Com apenas dois pontos de teste, não é possível avaliar a 
robustez do modelo sob diferentes condições de sazonalidade e volume de 
treino.

A validação walk-forward resolve esta limitação mediante cortes temporais 
múltiplos e deslizantes ao longo do histórico disponível: treina-se o 
modelo até o mês X, avalia-se nos 3 meses subsequentes; incrementa-se o 
corte (X + 6 meses) e repete-se o processo. Esta abordagem gera múltiplas 
janelas de teste independentes, cobrindo diferentes estações e fases da 
curva de crescimento do negócio.

### Escopo da comparação

A validação compara Baseline (SMA-3) e XGBoost em cada janela temporal, 
permitindo avaliar não apenas o desempenho médio do XGBoost, mas também 
a consistência de sua vantagem relativa e a identificação de padrões 
sazonais no ganho preditivo.

##Resultados - Etapa 11

### Escopo da validação

Foram avaliadas 9 janelas de 3 meses cada, distribuídas ao longo de 5 anos 
(2022-2026), cobrindo repetidamente ambas as estações (alta e baixa 
temporada).

### Resultados agregados

**Desempenho comparativo**: XGBoost superou o baseline em 8 das 9 janelas 
avaliadas, apresentando WAPE médio de 1.440 contra 1.605 do baseline 
(melhoria de ~10%). Adicionalmente, o desvio padrão do WAPE foi 
substancialmente menor (0.219 vs 0.400), indicando maior consistência e 
previsibilidade ao longo do tempo.

**Padrão sazonal identificado**: a vantagem do XGBoost concentra-se nas 
janelas de baixa temporada (jul/ago/set — inverno), com reduções de 15-22% 
no WAPE. Nas janelas de alta temporada (jan/fev/mar — verão), ambos os 
modelos apresentam desempenho equivalente. Este padrão manifesta-se 
consistentemente em múltiplos anos, descartando efeito de coincidência 
anual.

**Relação com volume de treino**: a única janela em que o baseline apresentou 
desempenho superior foi aquela com menor histórico de treino disponível 
(24 meses), sugerindo que a vantagem do XGBoost aumenta proporcionalmente 
à disponibilidade de dados históricos.

### Conclusão metodológica

XGBoost é confirmado como modelo recomendado para implementação, sustentado 
por evidência estatística robusta através de múltiplas janelas temporais 
independentes, descartando a hipótese de desempenho favorável por 
coincidência de período.

In [0]:
from pyspark.sql import functions as F

# Lista de cortes de treino, espaçados a cada 6 meses, cobrindo diferentes estações
# e diferentes pontos da curva de crescimento do negócio.
cutoffs = list(range(24, 78, 6))  # ex: [24, 30, 36, 42, 48, 54, 60, 66, 72]
print("Cortes (month_index) a testar:", cutoffs)


def calcular_baseline_janela(df_features, cutoff_month_index, test_months):
    """Mesma lógica de baseline da Etapa 6, encapsulada pra reuso em cada janela do walk-forward."""
    df_test = df_features.filter(F.col("year_month").isin(test_months))
    df_baseline_snapshot = (
        df_features
        .filter(F.col("month_index") == cutoff_month_index + 1)
        .select("variant_sk", "location_sk", F.col("roll_mean_3m").alias("baseline_sma3"))
    )
    df_test_baseline = (
        df_test
        .select("variant_sk", "location_sk", F.col("qty_month").alias("actual"))
        .join(df_baseline_snapshot, on=["variant_sk", "location_sk"], how="left")
        .withColumn("error", F.col("actual") - F.col("baseline_sma3"))
    )
    row = df_test_baseline.agg(
        F.round(F.avg(F.abs("error")), 3).alias("mae"),
        F.round(F.sqrt(F.avg(F.pow("error", 2))), 3).alias("rmse"),
        F.round(F.sum(F.abs("error")) / F.sum("actual"), 4).alias("wape"),
    ).collect()[0]
    return row["mae"], row["rmse"], row["wape"]


# df_walkforward_rows: 1 dict por janela testada, com métricas do baseline e do XGBoost lado a lado.
# Reaproveita treinar_e_avaliar_xgb (Etapa 10) e cat_mappings (calculado sobre df_features_clean inteiro).
resultados_walkforward = []

for cutoff in cutoffs:
    test_year_months = [
        r["year_month"] for r in
        df_month_index.filter((F.col("month_index") > cutoff) & (F.col("month_index") <= cutoff + 3))
        .orderBy("month_index").collect()
    ]

    # Baseline dessa janela
    mae_bl, rmse_bl, wape_bl = calcular_baseline_janela(df_features, cutoff, test_year_months)

    # XGBoost dessa janela (reaproveita a função da Etapa 10)
    _, _, metrics_xgb_janela = treinar_e_avaliar_xgb(
        df_features_clean, cutoff_month_index=cutoff, test_months=test_year_months, label=f"corte_{cutoff}"
    )
    row_xgb = metrics_xgb_janela.collect()[0]

    resultados_walkforward.append({
        "cutoff_month_index": cutoff,
        "test_months": ",".join(test_year_months),
        "mae_baseline": mae_bl, "rmse_baseline": rmse_bl, "wape_baseline": wape_bl,
        "mae_xgb": row_xgb["mae"], "rmse_xgb": row_xgb["rmse"], "wape_xgb": row_xgb["wape"],
    })
    print(f"Janela corte={cutoff} ({test_year_months}) -> baseline WAPE={wape_bl:.3f} | XGB WAPE={row_xgb['wape']:.3f}")

# df_walkforward: 1 linha por janela testada, com métricas do baseline e do XGBoost lado a lado
df_walkforward = spark.createDataFrame(resultados_walkforward)
display(df_walkforward)

# Resumo agregado: média e desvio padrão do WAPE em todas as janelas, pra cada abordagem
display(
    df_walkforward.agg(
        F.round(F.avg("wape_baseline"), 4).alias("wape_medio_baseline"),
        F.round(F.stddev("wape_baseline"), 4).alias("wape_desvio_baseline"),
        F.round(F.avg("wape_xgb"), 4).alias("wape_medio_xgb"),
        F.round(F.stddev("wape_xgb"), 4).alias("wape_desvio_xgb"),
    )
)

# Quantas janelas o XGBoost realmente supera o baseline (WAPE menor)?
n_janelas_xgb_venceu = df_walkforward.filter(F.col("wape_xgb") < F.col("wape_baseline")).count()
n_janelas_total = df_walkforward.count()
print(f"XGBoost superou o baseline em {n_janelas_xgb_venceu} de {n_janelas_total} janelas testadas")

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# === Painel 1: Walk-forward — MAE por janela (valores da Etapa 11) ===
# MAE = erro medio absoluto em UNIDADES vendidas (intuitivo: "erra X unidades em media")
pdf_wf = pd.DataFrame([
    {"cutoff": 24, "periodo": "jan-mar 2022", "mae_baseline": 2.680, "mae_xgb": 2.716},
    {"cutoff": 30, "periodo": "jul-set 2022", "mae_baseline": 1.519, "mae_xgb": 1.195},
    {"cutoff": 36, "periodo": "jan-mar 2023", "mae_baseline": 2.987, "mae_xgb": 2.972},
    {"cutoff": 42, "periodo": "jul-set 2023", "mae_baseline": 1.660, "mae_xgb": 1.394},
    {"cutoff": 48, "periodo": "jan-mar 2024", "mae_baseline": 3.315, "mae_xgb": 3.290},
    {"cutoff": 54, "periodo": "jul-set 2024", "mae_baseline": 1.843, "mae_xgb": 1.522},
    {"cutoff": 60, "periodo": "jan-mar 2025", "mae_baseline": 3.620, "mae_xgb": 3.556},
    {"cutoff": 66, "periodo": "jul-set 2025", "mae_baseline": 2.123, "mae_xgb": 1.785},
    {"cutoff": 72, "periodo": "jan-mar 2026", "mae_baseline": 4.007, "mae_xgb": 3.893},
])

# === Painel 2: Comparação de modelos por cenário (valores do resumo Etapa 6/7/10) ===
comparativo = pd.DataFrame([
    {"modelo": "Baseline\n(SMA-3)",  "Inverno": 2.799, "Verão": 3.810},
    {"modelo": "Baseline\n(SMA-6)",  "Inverno": 3.078, "Verão": 3.576},
    {"modelo": "Random Forest",       "Inverno": 2.144, "Verão": 3.813},
    {"modelo": "RF + categoria",      "Inverno": 2.173, "Verão": 3.805},
    {"modelo": "XGBoost",             "Inverno": 1.958, "Verão": 3.844},
])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Painel 1: Linhas — walk-forward ---
ax = axes[0]
ax.plot(pdf_wf["periodo"], pdf_wf["mae_baseline"], "o--", color="#d62728",
        label="Baseline (SMA-3)", linewidth=2, markersize=7)
ax.plot(pdf_wf["periodo"], pdf_wf["mae_xgb"], "s-", color="#1f77b4",
        label="XGBoost", linewidth=2, markersize=7)
ax.axhline(pdf_wf["mae_baseline"].mean(), color="#d62728", linestyle=":", alpha=0.4)
ax.axhline(pdf_wf["mae_xgb"].mean(), color="#1f77b4", linestyle=":", alpha=0.4)
ax.set_title("Walk-forward: MAE por janela (9 janelas, 2022–2026)", fontsize=13, fontweight="bold")
ax.set_xlabel("Janela de teste")
ax.set_ylabel("MAE (unidades)")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
ax.set_ylim(bottom=0)
ax.tick_params(axis="x", rotation=45)

# --- Painel 2: Barras agrupadas — cenários Inverno vs Verão ---
ax = axes[1]
x = np.arange(len(comparativo))
w = 0.35
bars_inv = ax.bar(x - w/2, comparativo["Inverno"], w, label="Inverno (jun/jul/ago)", color="#4a90d9")
bars_ver = ax.bar(x + w/2, comparativo["Verão"], w, label="Verão (dez/jan/fev)", color="#f5a623")

# Destacar XGBoost com borda verde
idx_xgb = comparativo[comparativo["modelo"].str.contains("XGBoost")].index[0]
for bar in [bars_inv[idx_xgb], bars_ver[idx_xgb]]:
    bar.set_edgecolor("green")
    bar.set_linewidth(2.5)

ax.set_title("Comparação de modelos: MAE por cenário", fontsize=13, fontweight="bold")
ax.set_ylabel("MAE (unidades)")
ax.set_xticks(x)
ax.set_xticklabels(comparativo["modelo"])
ax.legend()
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

# RESUMO - COMPARAÇÃO DOS MODELOS
## Comparação consolidada de todos os modelos avaliados

| Modelo | Cenário | MAE | RMSE | WAPE |
|---|---|---|---|---|
| Baseline (SMA-3) | Inverno (jun/jul/ago 2026) | 2.799 | 3.885 | 260.4% |
| Baseline (SMA-6) | Inverno (jun/jul/ago 2026) | 3.078 | 3.748 | 286.3% |
| Baseline (SMA-3) | Verão (dez/jan/fev 2026) | 3.810 | 5.470 | 114.6% |
| Baseline (SMA-6) | Verão (dez/jan/fev 2026) | 3.576 | 5.323 | 107.6% |
| Random Forest | Inverno (jun/jul/ago 2026) | 2.144 | 2.806 | 199.4% |
| Random Forest | Verão (dez/jan/fev 2026) | 3.813 | 4.894 | 114.7% |
| Random Forest + categoria | Inverno (jun/jul/ago 2026) | 2.173 | 2.818 | 202.1% |
| Random Forest + categoria | Verão (dez/jan/fev 2026) | 3.805 | 4.900 | 114.5% |
| **XGBoost** | **Inverno (jun/jul/ago 2026)** | **1.958** | **2.755** | **182.2%** |
| **XGBoost** | **Verão (dez/jan/fev 2026)** | **3.844** | **4.892** | **115.7%** |

### Walk-forward validation (9 janelas de 3 meses, 2022-2026): Baseline vs. XGBoost

| Métrica agregada | Baseline (SMA-3) | XGBoost |
|---|---|---|
| WAPE médio | 1.605 | **1.440** |
| Desvio padrão do WAPE | 0.400 | **0.219** |
| Janelas vencidas (de 9) | 1 | **8** |

### Modelo selecionado

**XGBoost foi selecionado como modelo final para implementação.** Este 
apresenta desempenho superior ao baseline e ao Random Forest no cenário de 
baixa temporada (inverno) e desempenho equivalente no cenário de alta 
temporada (verão), sem registro de degradação em nenhuma condição avaliada. 
A validação walk-forward, cobrindo 9 janelas temporais ao longo de 5 anos, 
confirma que a vantagem observada é consistente e estatisticamente robusta, 
descartando hipótese de desempenho favorável restrito a períodos específicos. 
A feature `category_name` foi experimentalmente avaliada e descartada por 
não apresentar ganho preditivo mensurável, preservando simplicidade do 
modelo sem compromisso de performance.

## Etapa 12 — Previsão para set/out/nov 2026 (XGBoost treinado com histórico completo)

### Desafio metodológico: previsão multi-período sem dados futuros

As validações anteriores (Etapas 6, 7, 10, 11) dispunham dos valores reais 
dos períodos de teste, utilizados exclusivamente para cálculo das métricas 
de erro, nunca como entrada do modelo. Para a previsão dos próximos 3 
meses (set/out/nov 2026), esses valores não estão disponíveis, configurando 
uma previsão genuinamente prospectiva. Este cenário expõe uma dependência 
crítica: as features `lag_1`, `lag_2`, `lag_3` e as médias móveis exigem 
valores de períodos anteriores que, para outubro e novembro, também são 
alvo de previsão (não observados).

### Solução adotada: previsão recursiva (multi-step forecasting)

A técnica padrão para este problema é a previsão recursiva: os períodos 
são previstos sequencialmente, utilizando as próprias previsões anteriores 
como pseudo-observações para cálculo dos lags subsequentes.

1. **Setembro/2026**: todos os lags são baseados em valores observados 
   (`lag_1` = agosto real, `lag_12` = setembro/2025 real). Previsão sem 
   propagação de incerteza.
2. **Outubro/2026**: `lag_1` corresponde à previsão de setembro (valor 
   não-observado), `lag_2` = agosto real, `lag_12` = outubro/2025 real.
3. **Novembro/2026**: `lag_1` = previsão de outubro, `lag_2` = previsão de 
   setembro, `lag_3` = agosto real, `lag_12` = novembro/2025 real.

### Consideração sobre propagação de incerteza

A previsão de novembro incorpora os erros cometidos nas previsões de setembro 
e outubro. É fenômeno documentado na literatura de forecasting que a 
incerteza aumenta proporcionalmente ao horizonte de previsão em abordagens 
recursivas.

## Nota — Interpretação do nível de granularidade das previsões

As previsões individuais (SKU × loja × mês) tendem a convergir para a média 
histórica observada para cada produto em cada período, refletindo a baixa 
importância da feature lag_12 (0,77%) e a natureza intermitente da demanda 
(documentada na Etapa 3). Este comportamento é esperado e não compromete 
a validade da previsão: em nível agregado (totais por produto, categoria 
ou mês), os erros individuais exibem comportamento compensatório, 
resultando em estimativas consideravelmente mais confiáveis.

**Recomendação de uso**: as previsões devem ser empregadas para planejamento 
agregado (aquisições, dimensionamento de capacidade), não como garantias 
determinísticas de venda por SKU/loja.

In [0]:
import xgboost as xgb_lib
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

pdf_history = (
    df_monthly_full
    .join(df_month_index, on="year_month", how="left")
    .select("variant_sk", "location_sk", "month_index", "qty_month")
    .toPandas()
)
print("Linhas no histórico inicial:", len(pdf_history))

pdf_combos = pdf_history[["variant_sk", "location_sk"]].drop_duplicates().reset_index(drop=True)
cat_mappings_final = {
    col: {val: i for i, val in enumerate(pdf_combos[col].unique())}
    for col in CATEGORICAL_COLS
}

pdf_train_final = df_features_clean.select(*(CATEGORICAL_COLS + NUMERIC_COLS + [LABEL_COL])).toPandas()
for col in CATEGORICAL_COLS:
    pdf_train_final[f"{col}_idx"] = pdf_train_final[col].map(cat_mappings_final[col])
feature_cols = [f"{c}_idx" for c in CATEGORICAL_COLS] + NUMERIC_COLS

modelo_final = xgb_lib.XGBRegressor(n_estimators=100, max_depth=6, random_state=42)
modelo_final.fit(pdf_train_final[feature_cols], pdf_train_final[LABEL_COL])
print("Modelo final treinado com", len(pdf_train_final), "linhas de histórico real.")

meses_futuros = [
    {"month_index": 81, "month_num": 9, "quarter": 3, "year_month": "202609"},
    {"month_index": 82, "month_num": 10, "quarter": 4, "year_month": "202610"},
    {"month_index": 83, "month_num": 11, "quarter": 4, "year_month": "202611"},
]

previsoes_finais = []

for mes_info in meses_futuros:
    mi = mes_info["month_index"]

    # CORREÇÃO DO BUG: em vez de procurar uma linha "month_index == mi" que ainda não
    # existe, pivotamos o histórico (produto x loja nas linhas, month_index nas colunas)
    # e consultamos DIRETAMENTE os valores reais/já previstos nos meses mi-1, mi-2, mi-3, mi-12.
    pivot = pdf_history.pivot_table(
        index=["variant_sk", "location_sk"], columns="month_index", values="qty_month", aggfunc="first"
    )
    pivot = pivot.reindex(pd.MultiIndex.from_frame(pdf_combos))  # garante todas as 6.054 combinações, na ordem certa

    def get_mes(offset):
        col = mi - offset
        return pivot[col] if col in pivot.columns else pd.Series(np.nan, index=pivot.index)

    lag_1, lag_2, lag_3, lag_12 = get_mes(1), get_mes(2), get_mes(3), get_mes(12)
    roll_mean_3m = pd.concat([get_mes(1), get_mes(2), get_mes(3)], axis=1).mean(axis=1)
    roll_mean_6m = pd.concat([get_mes(i) for i in range(1, 7)], axis=1).mean(axis=1)

    pdf_predict_mes = pdf_combos.copy()
    pdf_predict_mes["month_index"] = mi
    pdf_predict_mes["month_num"] = mes_info["month_num"]
    pdf_predict_mes["quarter"] = mes_info["quarter"]
    pdf_predict_mes["lag_1"] = lag_1.values
    pdf_predict_mes["lag_2"] = lag_2.values
    pdf_predict_mes["lag_3"] = lag_3.values
    pdf_predict_mes["lag_12"] = lag_12.values
    pdf_predict_mes["roll_mean_3m"] = roll_mean_3m.values
    pdf_predict_mes["roll_mean_6m"] = roll_mean_6m.values

    for col in CATEGORICAL_COLS:
        pdf_predict_mes[f"{col}_idx"] = pdf_predict_mes[col].map(cat_mappings_final[col])

    pdf_predict_mes["qty_month"] = modelo_final.predict(pdf_predict_mes[feature_cols]).clip(min=0)
    pdf_predict_mes["year_month"] = mes_info["year_month"]

    previsoes_finais.append(pdf_predict_mes[["variant_sk", "location_sk", "month_index", "year_month", "qty_month"]])

    pdf_history = pd.concat([
        pdf_history,
        pdf_predict_mes[["variant_sk", "location_sk", "month_index", "qty_month"]]
    ], ignore_index=True)

df_forecast_final = spark.createDataFrame(pd.concat(previsoes_finais, ignore_index=True))

print("Total de linhas previstas (deve ser 6054 x 3 = 18.162):", df_forecast_final.count())
display(df_forecast_final.orderBy("month_index").limit(10))

display(
    df_forecast_final
    .groupBy("year_month")
    .agg(F.round(F.sum("qty_month"), 0).alias("qty_total_prevista"))
    .orderBy("year_month")
)

In [0]:
CATALOG = "workspace"  
SCHEMA = "Gold" 

In [0]:
dim_products_lookup = spark.table(f"{CATALOG}.{SCHEMA}.dim_products").select("variant_sk", "product_name", "sku")
dim_location_lookup = spark.table(f"{CATALOG}.{SCHEMA}.dim_location").select("location_sk", "location_name", "location_type")

df_forecast_legivel = (
    df_forecast_final
    .join(dim_products_lookup, on="variant_sk", how="left")
    .join(dim_location_lookup, on="location_sk", how="left")
    .select(
        "product_name", "sku", "location_name", "location_type",
        "year_month", F.round("qty_month", 1).alias("qty_prevista")
    )
    .orderBy("product_name", "location_name", "year_month")
)

display(df_forecast_legivel)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import xgboost as xgb_lib
import pandas as pd
import numpy as np

# === Reconstrói o pipeline (variáveis perdidas após restartPython) ===

# 1. Tabelas source
AS_OF_DATE_SK = 20260831
df_sales = spark.table("workspace.gold.fct_sales_items").filter(F.col("date_sk") <= AS_OF_DATE_SK)
df_dates = spark.table("workspace.gold.dim_dates")
df_prod  = spark.table("workspace.gold.dim_products")
df_loc   = spark.table("workspace.gold.dim_location")

# 2. Grade mensal zero-fill (6054 combos x 80 meses)
sales_range = df_sales.select(F.min("date_sk").alias("mn"), F.max("date_sk").alias("mx")).collect()[0]
df_cal = (
    df_dates
    .filter((F.col("date_sk") >= sales_range["mn"]) & (F.col("date_sk") <= sales_range["mx"]))
    .withColumn("year_month", F.substring(F.col("date_sk").cast("string"), 1, 6))
    .select("year_month").distinct()
)
df_monthly = (
    df_sales.withColumn("year_month", F.substring(F.col("date_sk").cast("string"), 1, 6))
    .groupBy("variant_sk", "location_sk", "year_month").agg(F.sum("quantity").alias("qty_month"))
)
df_combos = df_sales.select("variant_sk", "location_sk").distinct()
df_monthly_full = (
    df_combos.crossJoin(df_cal)
    .join(df_monthly, on=["variant_sk", "location_sk", "year_month"], how="left")
    .fillna(0, subset=["qty_month"])
)

# 3. Feature engineering (lags, rolling means, calendário, tendência)
w = Window.partitionBy("variant_sk", "location_sk").orderBy("year_month")
df_month_index = (
    df_monthly_full.select("year_month").distinct()
    .withColumn("month_index", F.dense_rank().over(Window.orderBy("year_month")))
)
df_features = (
    df_monthly_full
    .withColumn("lag_1", F.lag("qty_month", 1).over(w))
    .withColumn("lag_2", F.lag("qty_month", 2).over(w))
    .withColumn("lag_3", F.lag("qty_month", 3).over(w))
    .withColumn("lag_12", F.lag("qty_month", 12).over(w))
    .withColumn("roll_mean_3m", F.avg("qty_month").over(w.rowsBetween(-3, -1)))
    .withColumn("roll_mean_6m", F.avg("qty_month").over(w.rowsBetween(-6, -1)))
    .withColumn("month_num", F.col("year_month").substr(5, 2).cast("int"))
    .withColumn("quarter", F.ceil(F.col("month_num") / 3))
    .join(F.broadcast(df_month_index), on="year_month", how="left")
)

# 4. Setup do modelo (mesmas features e parâmetros das Etapas 10/12)
CAT_COLS = ["variant_sk", "location_sk"]
NUM_COLS = ["lag_1", "lag_2", "lag_3", "lag_12", "roll_mean_3m", "roll_mean_6m", "month_num", "quarter", "month_index"]
LABEL = "qty_month"
feat_cols = [f"{c}_idx" for c in CAT_COLS] + NUM_COLS
df_clean = df_features.dropna(subset=NUM_COLS)

# 5. Codificação categórica (calculada sobre todo o dataset)
pdf_cats = df_clean.select(*CAT_COLS).distinct().toPandas()
cat_map = {col: {v: i for i, v in enumerate(pdf_cats[col].unique())} for col in CAT_COLS}

# 6. Modelo de teste -> MAE por produto (validação no último trimestre real: jun/jul/ago 2026)
cols = CAT_COLS + NUM_COLS + [LABEL]
pdf_tr = df_clean.filter(F.col("month_index") <= 77).select(*cols).toPandas()
pdf_te = df_clean.filter(F.col("year_month").isin(["202606", "202607", "202608"])).select(*cols).toPandas()
for c in CAT_COLS:
    pdf_tr[f"{c}_idx"] = pdf_tr[c].map(cat_map[c])
    pdf_te[f"{c}_idx"] = pdf_te[c].map(cat_map[c])

m_test = xgb_lib.XGBRegressor(n_estimators=100, max_depth=6, random_state=42)
m_test.fit(pdf_tr[feat_cols], pdf_tr[LABEL])
pdf_te["pred"] = m_test.predict(pdf_te[feat_cols]).clip(min=0)
pdf_te["abs_err"] = (pdf_te[LABEL] - pdf_te["pred"]).abs()

mae_prod = pdf_te.groupby("variant_sk")["abs_err"].mean().round(2).reset_index()
mae_prod.columns = ["variant_sk", "mae_historico"]

# 7. Modelo final (todo o histórico) -> previsão recursiva set/out/nov 2026
pdf_tr_final = df_clean.select(*cols).toPandas()
for c in CAT_COLS:
    pdf_tr_final[f"{c}_idx"] = pdf_tr_final[c].map(cat_map[c])
m_final = xgb_lib.XGBRegressor(n_estimators=100, max_depth=6, random_state=42)
m_final.fit(pdf_tr_final[feat_cols], pdf_tr_final[LABEL])

pdf_hist = (
    df_monthly_full.join(df_month_index, on="year_month", how="left")
    .select("variant_sk", "location_sk", "month_index", "qty_month").toPandas()
)
pdf_c = pdf_hist[["variant_sk", "location_sk"]].drop_duplicates().reset_index(drop=True)

futuros = [
    {"mi": 81, "mn": 9, "q": 3, "ym": "202609"},
    {"mi": 82, "mn": 10, "q": 4, "ym": "202610"},
    {"mi": 83, "mn": 11, "q": 4, "ym": "202611"},
]
prevs = []
for mf in futuros:
    mi = mf["mi"]
    piv = pdf_hist.pivot_table(index=["variant_sk", "location_sk"], columns="month_index", values="qty_month", aggfunc="first")
    piv = piv.reindex(pd.MultiIndex.from_frame(pdf_c))
    def gm(off):
        c = mi - off
        return piv[c] if c in piv.columns else pd.Series(np.nan, index=piv.index)
    pdf_p = pdf_c.copy()
    pdf_p["month_index"] = mi
    pdf_p["month_num"] = mf["mn"]
    pdf_p["quarter"] = mf["q"]
    pdf_p["lag_1"] = gm(1).values
    pdf_p["lag_2"] = gm(2).values
    pdf_p["lag_3"] = gm(3).values
    pdf_p["lag_12"] = gm(12).values
    pdf_p["roll_mean_3m"] = pd.concat([gm(1), gm(2), gm(3)], axis=1).mean(axis=1).values
    pdf_p["roll_mean_6m"] = pd.concat([gm(i) for i in range(1, 7)], axis=1).mean(axis=1).values
    for c in CAT_COLS:
        pdf_p[f"{c}_idx"] = pdf_p[c].map(cat_map[c])
    pdf_p["qty_month"] = m_final.predict(pdf_p[feat_cols]).clip(min=0)
    pdf_p["year_month"] = mf["ym"]
    prevs.append(pdf_p[["variant_sk", "location_sk", "year_month", "qty_month"]])
    pdf_hist = pd.concat([pdf_hist, pdf_p[["variant_sk", "location_sk", "month_index", "qty_month"]]], ignore_index=True)

# 8. Juntar MAE por produto + nomes legíveis na previsão final
pdf_fc = pd.concat(prevs, ignore_index=True)
pdf_fc = pdf_fc.merge(mae_prod, on="variant_sk", how="left")
pdf_fc = pdf_fc.merge(df_prod.select("variant_sk", "product_name", "sku").toPandas(), on="variant_sk", how="left")
pdf_fc = pdf_fc.merge(df_loc.select("location_sk", "location_name", "location_type").toPandas(), on="location_sk", how="left")
pdf_fc = pdf_fc[["product_name", "sku", "location_name", "location_type", "year_month", "qty_month", "mae_historico"]]
pdf_fc.columns = ["product_name", "sku", "location_name", "location_type", "year_month", "qty_prevista", "mae_historico"]
pdf_fc["qty_prevista"] = pdf_fc["qty_prevista"].round(1)
pdf_fc = pdf_fc.sort_values(["product_name", "location_name", "year_month"]).reset_index(drop=True)

display(spark.createDataFrame(pdf_fc))
print(f"Total: {len(pdf_fc)} previsões | MAE medio: {pdf_fc['mae_historico'].mean():.2f} | MAE mediano: {pdf_fc['mae_historico'].median():.2f}")

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Produtos solicitados (busca por nome parcial nos SKUs/nomes)
produtos_interesse = [
    "Motor de Popa 7678",
    "Sonar Transducer 4970",
    "Vela Mestra 1319",
    "Cabo Náutico 9083",
    "Cabo Náutico 8225"
]

# Carrega lookup de produtos (filtra nomes inválidos)
dim_products_lookup_pdf = (
    spark.table("workspace.gold.dim_products")
    .filter("product_name != 'asdf'")
    .select("variant_sk", "product_name")
    .toPandas()
)

# Filtra histórico dos últimos 12 meses reais (month_index 69-80 = set/2025 a ago/2026)
historico_12m = (
    pdf_hist[pdf_hist["month_index"].between(69, 80)]
    .merge(dim_products_lookup_pdf, on="variant_sk", how="left")
    .groupby(["product_name", "month_index"])["qty_month"]
    .sum()
    .reset_index()
)

# Adiciona lookup de year_month para os índices (inclui meses futuros 81-83)
month_idx_to_ym = df_month_index.toPandas()
# Adiciona meses futuros que não estão no histórico
meses_futuros_map = pd.DataFrame([
    {"year_month": "202609", "month_index": 81},
    {"year_month": "202610", "month_index": 82},
    {"year_month": "202611", "month_index": 83}
])
month_idx_to_ym = pd.concat([month_idx_to_ym, meses_futuros_map], ignore_index=True).drop_duplicates()
historico_12m = historico_12m.merge(month_idx_to_ym, on="month_index", how="left")

# Filtra previsões (set/out/nov 2026) - agrupa por produto
previsoes_3m = (
    pdf_fc[pdf_fc["product_name"].isin(produtos_interesse)]
    .groupby(["product_name", "year_month"])["qty_prevista"]
    .sum()
    .reset_index()
    .rename(columns={"qty_prevista": "qty_month"})
)

# Mapeia year_month para month_index nas previsões
previsoes_3m = previsoes_3m.merge(
    month_idx_to_ym,
    on="year_month",
    how="left"
)

# Combina histórico + previsões para cada produto
fig, axes = plt.subplots(len(produtos_interesse), 1, figsize=(14, 3 * len(produtos_interesse)))
if len(produtos_interesse) == 1:
    axes = [axes]

for idx, produto in enumerate(produtos_interesse):
    ax = axes[idx]
    
    # Dados históricos do produto
    hist_prod = historico_12m[historico_12m["product_name"] == produto].sort_values("month_index")
    
    # Previsões do produto
    prev_prod = previsoes_3m[previsoes_3m["product_name"] == produto].sort_values("month_index")
    
    if len(hist_prod) > 0:
        # Plota histórico
        ax.plot(
            hist_prod["month_index"],
            hist_prod["qty_month"],
            marker="o",
            linewidth=2,
            label="Histórico (Set/2025 - Ago/2026)",
            color="#1f77b4"
        )
        
        # Plota previsões
        if len(prev_prod) > 0:
            # Conecta último ponto do histórico com primeira previsão
            ultimo_hist = hist_prod.iloc[-1]
            primeiro_prev = prev_prod.iloc[0]
            
            ax.plot(
                [ultimo_hist["month_index"], primeiro_prev["month_index"]],
                [ultimo_hist["qty_month"], primeiro_prev["qty_month"]],
                linestyle="--",
                linewidth=1.5,
                color="gray",
                alpha=0.5
            )
            
            ax.plot(
                prev_prod["month_index"],
                prev_prod["qty_month"],
                marker="s",
                linewidth=2,
                linestyle="--",
                label="Previsão (Set/Out/Nov 2026)",
                color="#ff7f0e"
            )
            
            # Adiciona área sombreada nas previsões
            ax.axvspan(
                prev_prod["month_index"].min() - 0.5,
                prev_prod["month_index"].max() + 0.5,
                alpha=0.1,
                color="orange",
                label="Período de Previsão"
            )
        
        # Formata eixo X com os meses
        todos_indices = pd.concat([hist_prod[["month_index", "year_month"]], prev_prod[["month_index", "year_month"]]])
        todos_indices = todos_indices.drop_duplicates().sort_values("month_index")
        
        # Mostra apenas alguns labels para não poluir
        tick_df = todos_indices.drop_duplicates(subset="month_index").sort_values("month_index")
        indices_mostrar = tick_df["month_index"].values[::3]
        tick_df_filtrado = tick_df[tick_df["month_index"].isin(indices_mostrar)]
        
        ax.set_xticks(tick_df_filtrado["month_index"].values)
        ax.set_xticklabels(
            [f"{ym[:4]}/{ym[4:]}" for ym in tick_df_filtrado["year_month"].values],
            rotation=45, ha="right"
        )
        
        ax.set_title(f"{produto}", fontsize=12, fontweight="bold")
        ax.set_xlabel("Mês", fontsize=10)
        ax.set_ylabel("Quantidade Agregada", fontsize=10)
        ax.legend(loc="best", fontsize=9)
        ax.grid(True, alpha=0.3, linestyle=":")
    else:
        ax.text(0.5, 0.5, f"Sem dados para {produto}", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{produto}", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.suptitle(
    "Previsão de Demanda - Próximos 3 Meses (Set/Out/Nov 2026)",
    fontsize=14,
    fontweight="bold",
    y=1.002
)
plt.subplots_adjust(top=0.97)
plt.show()

# Exibe tabela resumo das previsões
print("\n=== RESUMO DAS PREVISÕES (Set/Out/Nov 2026) ===")
resumo = previsoes_3m.pivot_table(
    index="product_name",
    columns="year_month",
    values="qty_month",
    aggfunc="sum"
).round(1)
print(resumo)

##Etapa 13 - Seleção do Produto para Demonstração de Sazonalidade

 Em substituição à seleção empírica de um produto para ilustrar sazonalidade,
 foi calculada a **força da sazonalidade** de cada produto com histórico suficiente
 (mínimo de 36 meses), utilizando a decomposição clássica de série temporal
 (`statsmodels.tsa.seasonal.seasonal_decompose`):

 $$ F_{sazonal} = \max\left(0,\ 1 - \frac{Var(\text{resíduo})}{Var(\text{sazonal} + \text{resíduo})}\right) $$

 Valores próximos a 1 indicam padrão sazonal forte e regular, enquanto valores
 próximos a 0 indicam comportamento predominantemente aleatório, sem repetição
 anual consistente. Esta métrica permite a comparação objetiva de múltiplos
 produtos candidatos simultaneamente, substituindo a inspeção manual de gráficos
 individuais. O processo de lookup de produtos foi ajustado para filtrar apenas registros com
 `is_valid_product_name = true` antes do cálculo.



In [0]:
%pip install statsmodels
dbutils.library.restartPython()

In [0]:
# Filtra "asdf" e outros nomes inválidos de cadastro (dado sujo — não são produtos reais)
dim_products_lookup_pdf = (
    spark.table("workspace.gold.dim_products")
    .filter("product_name != 'asdf'")
    .select("variant_sk", "product_name")
    .toPandas()
)
dim_location_lookup_pdf = (
    spark.table("workspace.gold.dim_location")
    .select("location_sk", "location_name")
    .toPandas()
)

# pdf_c vem da célula "Previsão com MAE por produto" (equivalente ao pdf_combos da Etapa 12)
pdf_combos_com_nomes = (
    pdf_c
    .merge(dim_products_lookup_pdf, on="variant_sk", how="left")
    .merge(dim_location_lookup_pdf, on="location_sk", how="left")
)

In [0]:
from statsmodels.tsa.seasonal import seasonal_decompose
import pandas as pd
import numpy as np

# Reaproveita pdf_hist da célula "Previsão com MAE por produto" (filtrado até month_index 80 = ago/2026).
# Ideia: testar N produtos com bastante histórico e escolher o de sazonalidade mais forte.

CANDIDATOS_MIN_MESES = 36  # só considera produtos com pelo menos 3 anos de histórico

historico_produto = (
    pdf_hist[pdf_hist["month_index"] <= 80]
    .merge(pdf_combos_com_nomes, on=["variant_sk", "location_sk"], how="left")
    .groupby(["product_name", "month_index"])["qty_month"]
    .sum()
    .reset_index()
)

candidatos = (
    historico_produto.groupby("product_name")["month_index"]
    .count()
    .loc[lambda s: s >= CANDIDATOS_MIN_MESES]
    .index.tolist()
)

forcas = []
for produto in candidatos:
    serie = (
        historico_produto[historico_produto["product_name"] == produto]
        .sort_values("month_index")
        .set_index("month_index")["qty_month"]
    )
    try:
        resultado = seasonal_decompose(serie, model="additive", period=12, extrapolate_trend="freq")
        var_resid = np.nanvar(resultado.resid)
        var_seasonal_resid = np.nanvar(resultado.seasonal + resultado.resid)
        forca = max(0, 1 - var_resid / var_seasonal_resid) if var_seasonal_resid > 0 else 0
        forcas.append({"product_name": produto, "forca_sazonalidade": forca, "n_meses": len(serie)})
    except Exception as e:
        continue  # produto com histórico insuficiente ou série com problema, pula

ranking_sazonalidade = pd.DataFrame(forcas).sort_values("forca_sazonalidade", ascending=False)
ranking_sazonalidade.head(10)

##Produto escolhido: Bússola de Bordo 1854

 Maior força de sazonalidade entre os produtos válidos (**0.74**), com 83 meses
 de histórico (quase 7 anos), uma base robusta o suficiente pra confiar no
 resultado.

 A decomposição mostra três coisas com clareza:
 - **Tendência**: crescimento consistente ao longo dos anos (~15 → ~50 unidades),
   com aceleração nos últimos meses. O produto está ganhando mercado, além do
   efeito sazonal.
 - **Sazonalidade**: padrão extremamente regular, com amplitude praticamente
   constante (±20) repetindo a cada 12 meses.
 - **Resíduo**: disperso em torno de zero, sem padrão remanescente, indicando
   que tendência + sazonalidade já explicam a maior parte do comportamento da
   série.

In [0]:
import matplotlib.pyplot as plt

serie_bussola = (
    historico_produto[historico_produto["product_name"] == "Bússola de Bordo 1854"]
    .sort_values("month_index")
    .set_index("month_index")["qty_month"]
)

resultado_bussola = seasonal_decompose(serie_bussola, model="additive", period=12, extrapolate_trend="freq")

fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
resultado_bussola.observed.plot(ax=axes[0], title="Demanda observada")
resultado_bussola.trend.plot(ax=axes[1], title="Tendência")
resultado_bussola.seasonal.plot(ax=axes[2], title="Sazonalidade")
resultado_bussola.resid.plot(ax=axes[3], title="Resíduo", marker="o", linestyle="none")
plt.tight_layout()
plt.show()

In [0]:
sazonal_por_mes = resultado_bussola.seasonal.reset_index()
sazonal_por_mes.columns = ["month_index", "sazonal"]

month_index_lookup = df_month_index.select("month_index", "year_month").toPandas()

picos_sazonais = (
    sazonal_por_mes
    .merge(month_index_lookup, on="month_index", how="left")
    .assign(mes_calendario=lambda d: pd.to_datetime(d["year_month"], format="%Y%m").dt.month)
    .groupby("mes_calendario")["sazonal"]
    .mean()
    .sort_values(ascending=False)
)
picos_sazonais

## Conclusão e Resultados
 Traduzindo o pico sazonal médio por mês do calendário: os meses de maior venda
 são **dezembro, janeiro e fevereiro** (verão), e os de menor venda são
 **junho, agosto e setembro** (inverno).

 Esse padrão **coincide com o achado da validação walk-forward do XGBoost**
 (Etapa 11), que identificou exatamente esse mesmo trimestre de inverno
 (jul/ago/set) como o de menor atividade de vendas

 O `seasonal_decompose` devolve o componente sazonal indexado por `month_index`
 (posição sequencial no histórico, ex: mês 1, 2, 3...), que não diz nada sobre
 qual mês do ano cada posição representa. Para extrair uma leitura de negócio
 (qual mês do calendário puxa o pico), foi necessário:
 1. Buscar o `year_month` real correspondente a cada `month_index` via
    `df_month_index` (join pela chave, não por posição/ordem — evita descasar
    os dados caso as duas séries tenham tamanhos diferentes).
 2. Converter `year_month` (ex: `"202601"`) para o mês do calendário (`1`–`12`).
 3. Calcular a média do componente sazonal por mês do calendário, agregando
    os ~7 anos de histórico.

 Resultado:

 | Mês | Efeito sazonal médio |
 |---|---|
 | Janeiro | **+23.8** (pico) |
 | Fevereiro | +18.0 |
 | Março | +14.0 |
 | Dezembro | +13.8 |
 | Abril | +6.0 |
 | Outubro | +1.1 |
 | Novembro | -3.7 |
 | Maio | -8.2 |
 | Julho | -10.6 |
 | Setembro | -15.2 |
 | Junho | -18.8 |
 | Agosto | **-20.1** (vale) |

 O padrão confirma a hipótese de negócio: o trimestre dez-jan-fev (verão, alta
 temporada náutica) concentra o efeito sazonal positivo, e o trimestre jun-jul-ago
 (inverno, baixa temporada) concentra o efeito negativo com uma transição suave
 nos meses de entressafra (abril, outubro, novembro).

 Esse mesmo recorte de trimestres (verão x inverno) é o que já havia sido usado,
 de forma independente, na validação walk-forward do XGBoost (Etapa 11) — reforçando
 que não é uma coincidência estatística de um único produto, mas um padrão de
 calendário que se repete no comportamento agregado do negócio como um todo.

 Com isso, a pergunta está respondida por completo:
 previsão de demanda por produto/loja para os próximos 3 meses (`df_forecast_legivel`) e identificação e justificativa de sazonalidade em um produto escolhido
 (Bússola de Bordo 1854).

#Estudo de Caso 2 - Previsão de Demanda nos Centros de Distribuição

## Etapa 1 — Caracterização dos dados e teste da hipótese "CD = canal e-commerce exclusivo"

### Objetivo desta etapa

Antecedendo qualquer análise de crescimento ou recomendação de expansão, 
faz-se necessária a caracterização preliminar dos dados disponíveis: 
volume de produtos, localizações operacionais, base de clientes e sua 
distribuição geográfica. Adicionalmente, será testada a hipótese de que 
centros de distribuição atendem exclusivamente o canal e-commerce, ou se 
também processam transações presenciais e B2B.

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

# AS_OF_DATE_SK: corte OFICIAL, usado em toda análise que vai pro relatório/apresentação.
AS_OF_DATE_SK = 20260831

# AS_OF_DATE_SK_FULL: inclui o "futuro sintético" da base (até dez/2026).
# USO EXCLUSIVO para checagem interna -- nunca aparece em números/gráficos oficiais.
AS_OF_DATE_SK_FULL = 20261231

df_sales = (
    spark.table("workspace.gold.fct_sales_items")
    .filter(F.col("date_sk") <= AS_OF_DATE_SK)
    .withColumn("year_month", F.substring(F.col("date_sk").cast("string"), 1, 6))
)
df_sales_full = spark.table("workspace.gold.fct_sales_items").filter(F.col("date_sk") <= AS_OF_DATE_SK_FULL)

df_loc = spark.table("workspace.gold.dim_location")
df_prod = spark.table("workspace.gold.dim_products")
df_cust = spark.table("workspace.gold.dim_customers")

print(f"df_sales (oficial, até {AS_OF_DATE_SK}):", df_sales.count(), "linhas")
print(f"df_sales_full (checagem interna, até {AS_OF_DATE_SK_FULL}):", df_sales_full.count(), "linhas")

# === FICHA-RESUMO ===
print("="*60)
print("FICHA-RESUMO DOS DADOS")
print("="*60)
print(f"Produtos distintos:        {df_prod.select('variant_sk').distinct().count()}")
print(f"Locais (lojas + CDs):      {df_loc.count()}")
print(f"  - Lojas (store):         {df_loc.filter(F.col('location_type')=='store').count()}")
print(f"  - Centros de dist. (warehouse): {df_loc.filter(F.col('location_type')=='warehouse').count()}")
print(f"Clientes distintos:        {df_cust.select('customer_sk').distinct().count()}")
print(f"Estados com local (loja/CD): {df_loc.select('location_state').distinct().count()}")
print(f"Estados com clientes:      {df_cust.select('primary_state').distinct().count()}")
print(f"Cidades com clientes:      {df_cust.select('primary_city').distinct().count()}")
print("="*60)

display(df_loc.select("location_id", "location_name", "location_type", "city", "location_state"))

# === TESTE 1: composição de canal por TIPO de local (store vs warehouse) ===
df_channel_by_type = (
    df_sales
    .drop("location_id")
    .join(df_loc.select("location_sk", "location_type"), on="location_sk", how="left")
    .groupBy("location_type", "channel")
    .agg(F.count("*").alias("num_pedidos"), F.sum("quantity").alias("qty_total"))
    .orderBy("location_type", "channel")
)
display(df_channel_by_type)

pdf_channel = df_channel_by_type.toPandas()
pdf_pivot = pdf_channel.pivot(index="location_type", columns="channel", values="num_pedidos").fillna(0)
pdf_pivot_pct = pdf_pivot.div(pdf_pivot.sum(axis=1), axis=0) * 100

pdf_pivot_pct.plot(kind="barh", stacked=True, figsize=(9, 4), colormap="tab10")
plt.title("Composição de canal de venda por tipo de local (%)")
plt.xlabel("% dos pedidos")
plt.ylabel("Tipo de local")
plt.legend(title="Canal", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# === Clientes por estado (prévia) ===
df_cust_by_state = (
    df_cust
    .groupBy("primary_state")
    .agg(F.count("*").alias("num_clientes"))
    .orderBy(F.desc("num_clientes"))
)
display(df_cust_by_state)

pdf_cust_state = df_cust_by_state.toPandas()
plt.figure(figsize=(9, 5))
plt.barh(pdf_cust_state["primary_state"], pdf_cust_state["num_clientes"])
plt.gca().invert_yaxis()
plt.title("Clientes por estado (primary_state)")
plt.xlabel("Número de clientes")
plt.tight_layout()
plt.show()

## Resultados da Etapa 1 — Caracterização e teste de canal

### Caracterização da base

- Catálogo contendo **1.009 produtos**, operação em **6 locais** (3 lojas + 
  3 CDs), porém **2.000 clientes distribuídos em 27 estados** — primeira 
  evidência de concentração substancialmente maior da infraestrutura física 
  em relação à dispersão geográfica da base de clientes.
- Localizações físicas presentes em apenas **4 estados** (PR, AP, PI, AL) 
  — regiões Sul, Norte e Nordeste. As regiões Sudeste e Centro-Oeste 
  encontram-se inteiramente descobertas.

### Teste de hipótese: canal por tipo de local

- **Composição de canal idêntica entre `store` e `warehouse`** (70% 
  e-commerce, 30% POS em ambos) — ausência de diferenciação comportamental 
  por tipo de localização, indicando que o canal pode não refletir lógica 
  operacional real (hipótese confirmada na Etapa 3).

# Etapa 2 — Distribuição geográfica da demanda versus cobertura física

In [0]:
estados_cobertos = [row["location_state"] for row in df_loc.select("location_state").distinct().collect()]
print("Estados com local físico (loja ou CD):", estados_cobertos)

df_demand_by_state = (
    df_sales
    .join(df_cust.select("customer_sk", "primary_state", "primary_city"), on="customer_sk", how="left")
    .groupBy("primary_state")
    .agg(
        F.countDistinct("customer_sk").alias("num_clientes"),
        F.sum("quantity").alias("qty_total"),
        F.sum("net_sales_amount").alias("receita_liquida_total"),
    )
    .withColumn("coberto_hoje", F.col("primary_state").isin(estados_cobertos))
    .orderBy(F.desc("receita_liquida_total"))
)
display(df_demand_by_state)

pdf_demand = df_demand_by_state.toPandas()
colors = pdf_demand["coberto_hoje"].map({True: "#2ca02c", False: "#d62728"})

plt.figure(figsize=(9, 7))
plt.barh(pdf_demand["primary_state"], pdf_demand["receita_liquida_total"], color=colors)
plt.gca().invert_yaxis()
plt.title("Receita líquida por estado do cliente\n(verde = já tem local físico, vermelho = sem cobertura)")
plt.xlabel("Receita líquida total (R$)")
plt.tight_layout()
plt.show()

display(
    df_demand_by_state
    .groupBy("coberto_hoje")
    .agg(F.sum("receita_liquida_total").alias("receita_total"), F.sum("qty_total").alias("qty_total"))
)

## Resultados da Etapa 2 — Concentração geográfica da receita

### Distribuição da receita por cobertura física

**Estados cobertos** (PR, AP, PI, AL): R$ 204,5 milhões  
**Estados não cobertos** (23 estados restantes): R$ 1.135,4 milhões

### Achado principal

**85% da receita total da empresa (R$ 1,14 bi de R$ 1,34 bi) é gerada em 
estados sem infraestrutura física.** Este resultado constitui o argumento 
quantitativo central para a questão sobre expansão de centros de 
distribuição: a cobertura atual atende apenas fração minoritária das 
regiões geradoras de receita.

### Estados de maior receita entre os não cobertos

- **DF**: R$ 60M  
- **RJ**: R$ 57M  
- **AM**: R$ 56M

Nenhum desses três estados possui loja ou centro de distribuição atualmente.

### Correlação entre número de clientes e receita

A hierarquização por número de clientes (Etapa 1) e por receita gerada 
apresentam consistência, sem discrepâncias significativas do tipo 
"concentração extrema em poucos clientes de alto valor" ou "pulverização 
em muitos clientes de baixo valor".

# Etapa 3 — Validação: canal reflete presença física real?

## Resultados da Etapa 3 — Hipótese do canal refutada

### Teste de correspondência geográfica por canal

Testou-se se o canal `pos` implica presença física do cliente no estado da 
localização (hipótese de compra presencial ou BOPIS — Buy Online, Pick-up 
In Store).

**Resultado:**  
- **POS**: 3,8% de correspondência entre estado do cliente e estado da loja  
- **E-commerce**: 3,9% de correspondência

Ambos os percentuais são estatisticamente equivalentes ao esperado por 
distribuição aleatória uniforme (1 em 27 estados ≈ 3,7%).

### Conclusão metodológica

O campo `channel` constitui atribuição arbitrária no dado sintético e não 
reflete padrão operacional real. Deve ser documentado como limitação da 
base de dados e não empregado como critério para decisões de expansão 
geográfica.

In [0]:
# df_channel_geo_check: compara estado do cliente com estado do local que processou
# a venda, por canal -- testa se 'pos' implica presença física real (BOPIS-like)
# ou se a atribuição de canal é arbitrária no dado sintético.
df_channel_geo_check = (
    df_sales
    .drop("location_id")
    .join(df_loc.select("location_sk", "location_state"), on="location_sk", how="left")
    .join(df_cust.select("customer_sk", "primary_state"), on="customer_sk", how="left")
    .withColumn("estado_bate", F.col("location_state") == F.col("primary_state"))
    .groupBy("channel")
    .agg(F.round(F.avg(F.col("estado_bate").cast("int")) * 100, 1).alias("pct_estado_bate"))
)
display(df_channel_geo_check)

# Etapa 4 — Análise de tendência, recorrência e ticket médio por estado

## Resultados da Etapa 4 — Métricas de diferenciação por estado

### Recorrência e ticket médio

- **Recorrência**: 100% em todos os 27 estados  
- **Ticket médio**: faixa estreita entre R$ 27,8k e R$ 30,2k

Ambas as métricas não diferenciam estados, provavelmente em decorrência das 
características da base sintética (7 anos de histórico + apenas 2.000 
clientes torna altamente provável que todo cliente tenha realizado múltiplas 
compras). Estas métricas foram avaliadas e descartadas como critérios de 
diferenciação.

### Crescimento como métrica diferenciadora

**Crescimento (últimos 12 meses vs. 12 meses anteriores)** constitui a 
métrica com maior poder discriminante, apresentando variação de +2,4% (MA) 
a +27,4% (SC).

**Maiores taxas de crescimento entre estados não cobertos:**  
- **SC**: +27,4%  
- **MS**: +27,3%  
- **BA**: +25,1%

Estes estados constituem candidatos por "momentum" de crescimento, em 
contraste com os candidatos por "tamanho de mercado" identificados na 
Etapa 2 (DF, RJ, AM).

### Justificativa para análise bidimensional

A identificação de duas dimensões complementares (volume absoluto de receita 
vs. taxa de crescimento) justifica a análise cruzada subsequente via gráfico 
de quadrantes.

In [0]:
df_sales_com_cliente = (
    df_sales
    .join(df_cust.select("customer_sk", "primary_state"), on="customer_sk", how="left")
)

# --- 1. TENDÊNCIA: últimos 12m (set/2025-ago/2026) vs 12m anteriores (set/2024-ago/2025) ---
df_trend_by_state = (
    df_sales_com_cliente
    .withColumn(
        "periodo",
        F.when(F.col("year_month") >= "202509", "ultimos_12m")
         .when((F.col("year_month") >= "202409") & (F.col("year_month") < "202509"), "12m_anteriores")
    )
    .filter(F.col("periodo").isNotNull())
    .groupBy("primary_state", "periodo")
    .agg(F.sum("net_sales_amount").alias("receita"))
    .groupBy("primary_state")
    .pivot("periodo", ["12m_anteriores", "ultimos_12m"])
    .agg(F.first("receita"))
    .withColumn("crescimento_pct", F.round((F.col("ultimos_12m") / F.col("12m_anteriores") - 1) * 100, 1))
)

# --- 2. RECORRÊNCIA: % de clientes com mais de 1 pedido, por estado ---
df_recorrencia_by_state = (
    df_sales_com_cliente
    .groupBy("primary_state", "customer_sk")
    .agg(F.countDistinct("order_id").alias("pedidos_do_cliente"))
    .groupBy("primary_state")
    .agg(
        F.count("*").alias("total_clientes"),
        F.sum((F.col("pedidos_do_cliente") > 1).cast("int")).alias("clientes_recorrentes"),
    )
    .withColumn("pct_recorrencia", F.round(F.col("clientes_recorrentes") / F.col("total_clientes") * 100, 1))
)

# --- 3. TICKET MÉDIO por estado ---
df_ticket_by_state = (
    df_sales_com_cliente
    .groupBy("primary_state")
    .agg(
        F.sum("net_sales_amount").alias("receita_total"),
        F.countDistinct("order_id").alias("total_pedidos"),
    )
    .withColumn("ticket_medio", F.round(F.col("receita_total") / F.col("total_pedidos"), 2))
)

df_estado_completo = (
    df_trend_by_state
    .join(df_recorrencia_by_state.select("primary_state", "pct_recorrencia"), on="primary_state", how="left")
    .join(df_ticket_by_state.select("primary_state", "ticket_medio", "receita_total"), on="primary_state", how="left")
    .withColumn("coberto_hoje", F.col("primary_state").isin(estados_cobertos))
    .orderBy(F.desc("crescimento_pct"))
)
display(df_estado_completo)

## Etapa 5 — Análise por quadrante: receita versus crescimento

### Justificativa metodológica

A representação bidimensional cruza as duas variáveis de maior relevância 
para decisão de expansão: volume de receita (tamanho da oportunidade) e 
taxa de crescimento (momentum de mercado). Estados posicionados no quadrante 
superior direito (alta receita + alto crescimento) sem cobertura física 
representam os candidatos prioritários. Estados no quadrante inferior 
esquerdo (baixa receita + baixo crescimento) constituem menor prioridade 
estratégica.

## Resultados da Etapa 5 — Identificação de candidatos prioritários

### Candidatos de alta prioridade (quadrante superior direito)

**RJ, PB e CE** combinam receita elevada já comprovada com crescimento 
acima da média — perfil que justifica investimento em nova infraestrutura 
física.

- **RJ**: 2º maior mercado (R$ 56,9M) com crescimento de 16,2%, sem cobertura 
  física atual.
- **PB** e **CE**: receita e crescimento consistentes, ambos descobertos.

### Estados de alta receita com crescimento moderado

**DF** constitui o maior mercado absoluto (R$ 60M), porém apresenta o menor 
crescimento entre os grandes mercados não cobertos (5,7%). Candidato por 
volume, mas com indicação de possível maturação ou saturação que demanda 
investigação adicional.

### Estados de alto crescimento com receita moderada

**SC e MS** lideram em taxa de crescimento (~27%), porém com receita ainda 
moderada em termos absolutos. Constituem "apostas de momentum" para 
monitoramento futuro, não prioridade imediata.

### Posicionamento dos estados já cobertos

Os 4 estados com cobertura física atual (AL, AP, PI, PR) ocupam posições 
medianas no quadrante — nenhum apresenta valores extremos em receita ou 
crescimento, reforçando que a alocação histórica da infraestrutura física 
não priorizou as regiões de maior retorno potencial.

| Quadrante (posição no gráfico) | Interpretação | Estados identificados |
|---|---|---|
| Direita + Acima da média | Receita elevada e crescimento elevado — candidato prioritário | RJ, PB, CE |
| Direita + Abaixo da média | Receita elevada com crescimento moderado — mercado maduro | DF, AM, MA |
| Esquerda + Acima da média | Receita moderada com crescimento acelerado — monitoramento futuro | SC, MS, BA |
| Esquerda + Abaixo da média | Receita moderada e crescimento moderado — menor prioridade | AC, MG, PA |

In [0]:
import matplotlib.pyplot as plt

pdf_quad = df_estado_completo.select("primary_state", "receita_total", "crescimento_pct", "coberto_hoje").toPandas()

media_receita = pdf_quad["receita_total"].mean()
media_crescimento = pdf_quad["crescimento_pct"].mean()

plt.figure(figsize=(11, 8))
cores = pdf_quad["coberto_hoje"].map({True: "#2ca02c", False: "#d62728"})
plt.scatter(pdf_quad["receita_total"], pdf_quad["crescimento_pct"], c=cores, s=120, edgecolor="black")

for _, row in pdf_quad.iterrows():
    plt.annotate(row["primary_state"], (row["receita_total"], row["crescimento_pct"]),
                 xytext=(5, 5), textcoords="offset points", fontsize=9)

plt.axvline(media_receita, color="gray", linestyle="--", linewidth=1)
plt.axhline(media_crescimento, color="gray", linestyle="--", linewidth=1)
plt.title("Quadrante de decisão: Receita total x Crescimento (%)\n(verde = já coberto, vermelho = sem cobertura)")
plt.xlabel("Receita total (R$)")
plt.ylabel("Crescimento % (últimos 12m vs 12m anteriores)")
plt.tight_layout()
plt.show()

# Síntese da análise até a Etapa 5

## Questões de negócio endereçadas

1. Os centros de distribuição atuais apresentam crescimento? Existe 
   diferenciação de desempenho?
2. Há justificativa para abertura de novo centro de distribuição? Em qual 
   localização?

## Evidências identificadas

**85% da receita total da empresa (R$ 1,14 bi de R$ 1,34 bi) é gerada em 
estados sem infraestrutura física.** A cobertura atual abrange 4 estados 
(PR, AP, PI, AL), enquanto a base de clientes distribui-se por 27 estados.

Entre os estados descobertos, **RJ, PB e CE** destacam-se pela combinação de:
- Alto volume de receita já comprovado
- Crescimento acima da média nos últimos 12 meses

Estes mercados demonstram demanda consolidada e aceleração sustentada, mesmo 
sem suporte logístico localizado.

## Conclusão preliminar

Existe oportunidade concreta de expansão geográfica, priorizando RJ, PB e 
CE como primeiros candidatos para novo centro de distribuição.

## Caminho metodológico percorrido

1. Caracterização da cobertura física (6 locais em 4 estados) versus 
   distribuição da base de clientes (27 estados) — Etapa 1
2. Quantificação da receita fora da cobertura atual: 85% — Etapa 2
3. Teste e descarte do campo `channel` como critério decisório (não reflete 
   padrão operacional real) — Etapa 3

## Etapa 6 — Avaliação dos centros de distribuição existentes

### Escopo desta etapa

As etapas anteriores (1-5) mensuraram crescimento por **estado do cliente**, 
respondendo à questão "onde a demanda está crescendo" (relevante para 
identificação de novos mercados). Esta etapa endereça questão complementar: 
**qual o desempenho de crescimento dos 3 centros de distribuição já 
operacionais** (Camargo/PR, Siqueira/AP, Novaes/PI)?

A análise foca nas localizações físicas em si, não na geografia dos 
clientes atendidos.

### Questões estratégicas derivadas

Além de quantificar taxas de crescimento relativas, esta etapa permite 
identificar eventuais operações em estagnação ou retroção, informação 
relevante para decisões de:
- Manutenção da operação atual
- Realocação de recursos entre CDs
- Descontinuação de operações não justificadas

## Resultados da Etapa 6 — Desempenho dos CDs existentes

### Status operacional dos 3 centros de distribuição

Todos os centros de distribuição atuais apresentam crescimento consistente, 
sem sinais de estagnação ou retroção:

- **Camargo/PR**: +20,5%  
- **Novaes/PI**: +18,9%  
- **Siqueira/AP**: +17,4%

Nenhuma operação justifica redução ou descontinuação.

### Diferenciação entre CDs atuais

**Camargo (PR)** apresenta maior taxa de crescimento entre os três, porém a 
diferenciação é marginal (~3 pontos percentuais). Não há operação 
"problemática" destoando significativamente.

### Comparação com candidatos a expansão

O crescimento dos CDs atuais (17-20%) é comparável a RJ (16%) e PB (17%). 
**CE (23%) apresenta crescimento superior a qualquer CD existente**, 
reforçando sua posição como candidato prioritário.

### Conclusão

Não há justificativa para fechamento ou redução de nenhum CD atual. A 
operação existente encontra-se saudável. A oportunidade estratégica reside 
na expansão de cobertura geográfica, não na correção da base instalada.

In [0]:
from pyspark.sql import functions as F

# df_growth_by_cd: crescimento dos 3 armazéns já existentes, últimos 12m vs 12m
# anteriores -- mesma lógica de tendência da Etapa 4, agora no grão de LOCAL
# FÍSICO (location_id), não de estado do cliente.
df_growth_by_cd = (
    df_sales
    .drop("location_id")
    .join(df_loc.filter(F.col("location_type") == "warehouse").select("location_sk", "location_id", "location_name", "city", "location_state"), on="location_sk", how="inner")
    .withColumn(
        "periodo",
        F.when(F.col("year_month") >= "202509", "ultimos_12m")
         .when((F.col("year_month") >= "202409") & (F.col("year_month") < "202509"), "12m_anteriores")
    )
    .filter(F.col("periodo").isNotNull())
    .groupBy("location_id", "location_name", "city", "location_state", "periodo")
    .agg(F.sum("net_sales_amount").alias("receita"), F.sum("quantity").alias("qty"))
    .groupBy("location_id", "location_name", "city", "location_state")
    .pivot("periodo", ["12m_anteriores", "ultimos_12m"])
    .agg(F.first("receita"), F.first("qty"))
)

# Renomeia colunas do pivot (o pivot com 2 aggs gera nomes compostos automáticos)
df_growth_by_cd = (
    df_growth_by_cd
    .withColumnRenamed("12m_anteriores_first(receita)", "receita_12m_anteriores")
    .withColumnRenamed("ultimos_12m_first(receita)", "receita_ultimos_12m")
    .withColumnRenamed("12m_anteriores_first(qty)", "qty_12m_anteriores")
    .withColumnRenamed("ultimos_12m_first(qty)", "qty_ultimos_12m")
    .withColumn("crescimento_receita_pct", F.round((F.col("receita_ultimos_12m") / F.col("receita_12m_anteriores") - 1) * 100, 1))
    .withColumn("crescimento_qty_pct", F.round((F.col("qty_ultimos_12m") / F.col("qty_12m_anteriores") - 1) * 100, 1))
    .orderBy(F.desc("crescimento_receita_pct"))
)
display(df_growth_by_cd)

# Gráfico: crescimento de receita, um CD por barra -- visual direto pro Caua
import matplotlib.pyplot as plt

pdf_cd = df_growth_by_cd.toPandas()
cores_cd = ["#2ca02c" if x > 0 else "#d62728" for x in pdf_cd["crescimento_receita_pct"]]

plt.figure(figsize=(8, 4))
plt.barh(pdf_cd["location_name"], pdf_cd["crescimento_receita_pct"], color=cores_cd)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Crescimento de receita por CD (últimos 12m vs 12m anteriores)")
plt.xlabel("Crescimento %")
plt.tight_layout()
plt.show()

### Estrutura das etapas subsequentes

**Etapa 6** (concluída)  
Crescimento dos CDs atuais

**Etapa 7** (visualização complementar)  
Representação geográfica: CDs atuais e estados candidatos, proporcionando 
dimensão territorial da análise

**Etapa 8** (síntese final)  
Recomendação consolidada: decisão sobre manutenção, expansão ou 
redução de CDs, integrando achados das Etapas 1-7

## Etapa 7 — Representação geográfica da distribuição e cobertura

### Justificativa para visualização espacial

As Etapas 5 e 6 identificaram candidatos prioritários e confirmaram saúde 
operacional dos CDs atuais mediante métricas quantitativas e gráficos de 
barra/dispersão. A dimensão territorial, entretanto, permanece implícita 
nessas representações.

A visualização geográfica aproximada evidencia:
- Concentração dos 3 CDs atuais em faixa Norte/Nordeste/Sul
- Distribuição dos candidatos prioritários (CE, RJ, PB) e grandes mercados 
  descobertos (DF, SP) em outras regiões

Esta representação reforça visualmente o argumento de "cobertura territorial 
insuficiente" quantificado na Etapa 2.

## Resultados da Etapa 7 — Interpretação da visualização geográfica

### Confirmação visual dos achados quantitativos

A representação espacial corrobora os achados quantitativos anteriores: os 
3 CDs atuais distribuem-se nas bordas do território nacional (Norte, 
Nordeste interior, Sul), enquanto os candidatos prioritários (CE, PB, RJ) 
ocupam faixa territorial distinta, com menor cobertura.

### Identificação de sinergia logística potencial

O **Armazém Novaes (PI)** encontra-se geograficamente próximo a CE e PB, 
sugerindo que expansão nessa região pode aproveitar sinergia logística com 
infraestrutura existente, reduzindo custos de implantação e operação em 
comparação a operação totalmente isolada.

### RJ como expansão de novo território

**RJ** constitui o candidato geograficamente mais distante de qualquer CD 
atual, representando abertura de território genuinamente novo e ampliação 
mais significativa da cobertura nacional.

### Distribuição dos CDs atuais

Os 3 CDs (Amapá no norte, Piauí no nordeste, Paraná no sul) apresentam 
dispersão aparentemente equilibrada entre macrorregiões. Entretanto, nenhum 
se posiciona no eixo central/litoral onde concentram-se os candidatos 
prioritários (CE, PB, RJ), evidenciando descontinuidade territorial da 
cobertura.

### Candidatos de crescimento acelerado

Estados categorizados como "Monitorar" (DF, SC, MS) e "Sem destaque" 
apresentam dispersão espacial sem padrão geográfico coerente, conforme 
esperado, dado que a categorização baseou-se em métricas de crescimento e 
receita, não em proximidade geográfica.

### Nota metodológica

Representação utiliza coordenadas aproximadas (capital ou centro geográfico 
do estado), não geocodificação real de CEP. Suficiente para ilustração 
conceitual em notebook e apresentação, porém não substitui visualização de 
Business Intelligence com dados geoespaciais precisos.

In [0]:
import matplotlib.pyplot as plt

coords_estados = {
    "AC": (-9.97, -67.81), "AL": (-9.65, -35.73), "AP": (0.03, -51.07), "AM": (-3.10, -60.02),
    "BA": (-12.97, -38.51), "CE": (-3.72, -38.54), "DF": (-15.79, -47.88), "ES": (-20.32, -40.34),
    "GO": (-16.68, -49.25), "MA": (-2.53, -44.30), "MT": (-15.60, -56.10), "MS": (-20.44, -54.65),
    "MG": (-19.92, -43.94), "PA": (-1.46, -48.50), "PB": (-7.12, -34.86), "PR": (-25.43, -49.27),
    "PE": (-8.05, -34.90), "PI": (-5.09, -42.80), "RJ": (-22.91, -43.17), "RN": (-5.79, -35.21),
    "RS": (-30.03, -51.23), "RO": (-8.76, -63.90), "RR": (2.82, -60.67), "SC": (-27.60, -48.55),
    "SP": (-23.55, -46.63), "SE": (-10.91, -37.07), "TO": (-10.25, -48.32),
}

pdf_map = df_estado_completo.select("primary_state", "receita_total", "crescimento_pct", "coberto_hoje").toPandas()

# CORREÇÃO: receita_total vem como decimal.Decimal do Spark -- força conversão
# para float, senão a divisão por 2e5 (float) quebra com TypeError.
pdf_map["receita_total"] = pdf_map["receita_total"].astype(float)

pdf_map["lat"] = pdf_map["primary_state"].map(lambda s: coords_estados.get(s, (None, None))[0])
pdf_map["lon"] = pdf_map["primary_state"].map(lambda s: coords_estados.get(s, (None, None))[1])

prioritarios = ["CE", "RJ", "PB"]
monitorar = ["DF", "SC", "MS"]

def categorizar(row):
    if row["coberto_hoje"]:
        return "CD atual"
    elif row["primary_state"] in prioritarios:
        return "Prioridade de expansão"
    elif row["primary_state"] in monitorar:
        return "Monitorar"
    else:
        return "Sem destaque"

pdf_map["categoria"] = pdf_map.apply(categorizar, axis=1)
cores_categoria = {"CD atual": "#2ca02c", "Prioridade de expansão": "#d62728", "Monitorar": "#ff7f0e", "Sem destaque": "#c7c7c7"}

pdf_cds = df_loc.filter(F.col("location_type") == "warehouse").select("location_name", "location_state").toPandas()
pdf_cds["lat"] = pdf_cds["location_state"].map(lambda s: coords_estados.get(s, (None, None))[0])
pdf_cds["lon"] = pdf_cds["location_state"].map(lambda s: coords_estados.get(s, (None, None))[1])

plt.figure(figsize=(9, 10))
for cat, cor in cores_categoria.items():
    subset = pdf_map[pdf_map["categoria"] == cat]
    plt.scatter(
        subset["lon"], subset["lat"],
        s=subset["receita_total"] / 2e5,
        c=cor, label=cat, alpha=0.75, edgecolor="black",
    )
for _, row in pdf_map.iterrows():
    plt.annotate(row["primary_state"], (row["lon"], row["lat"]), fontsize=8, ha="center", va="center")

plt.scatter(pdf_cds["lon"], pdf_cds["lat"], marker="X", s=200, c="black", label="Local físico (CD)", zorder=5)

plt.title("Mapa aproximado: receita por estado, CDs atuais e candidatos a expansão\n(tamanho da bolha = receita total)")
plt.xlabel("Longitude (aproximada)")
plt.ylabel("Latitude (aproximada)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Etapa 8 — Recomendação consolidada: Centros de Distribuição

## Questões de negócio endereçadas
- **(c)** Qual o status de crescimento dos centros de distribuição atuais? 
  Há necessidade de ajustes (expansão, redução ou descontinuação)?
- **(d)** Existe justificativa para abertura de novo centro de distribuição? 
  Em qual localização?

Esta etapa consolida os achados das Etapas 1-7 em recomendação estratégica 
única.

---

## Questão (c): Status dos 3 CDs atuais

### Recomendação: MANTER todos os CDs existentes

| CD | Estado | Crescimento (12m) | Avaliação |
|---|---|---|---|
| Armazém Camargo | PR | +20,5% | Manter — maior taxa entre os 3 |
| Armazém Novaes | PI | +18,9% | Manter |
| Armazém Siqueira | AP | +17,4% | Manter |

Todos os centros de distribuição apresentam crescimento consistente, sem 
sinais de estagnação ou retroção (documentado na Etapa 6). Não existe 
justificativa para fechamento, redução ou realocação de recursos.

---

## Questão (d): Justificativa para novo CD

### Diagnóstico: cobertura territorial insuficiente

A infraestrutura física atual (6 localizações) atende apenas **4 estados** 
(PR, AP, PI, AL), enquanto a base de clientes distribui-se por **27 estados**.

**85% da receita total da empresa (R$ 1,14 bi de R$ 1,34 bi) é gerada fora 
da cobertura física atual** (Etapa 2).

A operação existente encontra-se saudável, porém apresenta alcance 
territorial substancialmente inferior à dimensão real do mercado atendido.

---

## Matriz de priorização para novo CD

| Estado | Receita | Crescimento (12m) | vs. média CDs atuais (~19%) | Classificação |
|---|---|---|---|---|
| **CE** | R$ 52,1M | **+23,0%** | Superior | **Prioridade 1** |
| **RJ** | R$ 56,9M | +16,2% | Levemente inferior | **Prioridade 2** |
| **PB** | R$ 55,5M | +17,2% | Levemente inferior | **Prioridade 3** |
| DF | R$ 60,0M | +5,7% | Substancialmente inferior | Monitoramento |
| SC / MS | R$ 41-47M | +27,4% / +27,3% | Superior | Monitoramento |

### Fundamentação da hierarquização

**Ceará (Prioridade 1)**
- Único candidato com crescimento superior a todos os CDs existentes (23% 
  vs. 17-20%)
- Combina volume relevante com maior momentum entre candidatos
- Proximidade geográfica ao Armazém Novaes (PI) sugere sinergia logística 
  potencial, conforme identificado na Etapa 7

**Rio de Janeiro (Prioridade 2)**
- 2º maior mercado em receita absoluta, sem cobertura física
- Candidato geograficamente mais isolado de qualquer CD atual
- Representa abertura de território genuinamente novo, ampliando 
  significativamente cobertura nacional

**Paraíba (Prioridade 3)**
- Perfil equilibrado entre volume e crescimento
- Proximidade geográfica ao Armazém Novaes (PI), similar ao Ceará

### Estados em monitoramento (não-prioritários)

**DF**: maior mercado em receita absoluta, porém apresenta menor crescimento 
entre candidatos relevantes (5,7%). Possível maturação de mercado ou 
dependência de fatores externos não contemplados nesta análise. Requer 
investigação adicional antes de priorização.

**SC/MS**: lideram em crescimento percentual, porém com base de receita ainda 
moderada. Constituem candidatos de longo prazo, a reavaliar em 1-2 anos 
conforme evolução da base.

---

## Recomendação estratégica consolidada

### Questão (c): CDs atuais

**Manter os 3 centros de distribuição existentes sem alterações.** Todos 
apresentam crescimento saudável, sem necessidade de ajustes operacionais.

### Questão (d): Novo CD

**Recomenda-se abertura de novo centro de distribuição, priorizando Ceará 
como primeira opção**, seguido por Rio de Janeiro e Paraíba.

Os três estados combinam:
- Volume de receita já comprovado
- Crescimento consistente
- Ausência atual de suporte logístico localizado

Ceará e Paraíba apresentam vantagem adicional de proximidade geográfica ao 
Armazém Novaes (PI), com potencial de sinergia logística. Rio de Janeiro 
representa expansão territorial mais significativa, sendo o candidato 
geograficamente mais isolado da infraestrutura atual.

Distrito Federal, Santa Catarina e Mato Grosso do Sul devem ser monitorados 
como candidatos de médio prazo, não constituindo prioridade imediata pelas 
razões fundamentadas acima.

## Limitações metodológicas

### Escopo da análise

- A recomendação baseia-se em **identificação de demanda que justifica 
  investimento**, não constituindo cálculo completo de viabilidade 
  econômica-financeira
- Não foram incorporados: custos de abertura/operação de CD, distâncias 
  logísticas exatas, ou análise de sensibilidade de preço/custo
- Representação geográfica (Etapa 7) utiliza coordenadas aproximadas 
  (capital/centro do estado), não geocodificação precisa de CEP

### Variáveis descartadas

- **Canal de venda** (`channel`): testado e descartado por não refletir 
  padrão operacional real nos dados (Etapa 3)
- **Recorrência de cliente e ticket médio**: não apresentaram poder 
  discriminante entre estados (Etapa 4), possivelmente devido a 
  características da base de dados sintética

# Resumo Executivo — Análise de Centros de Distribuição (LH Nautical)

## Escopo da análise

Análise estruturada em 8 etapas, baseada em dados gold (`fct_sales_items`, 
`dim_location`, `dim_customers`), endereçando duas questões estratégicas:
1. Status e necessidade de ajustes nos centros de distribuição atuais
2. Justificativa e localização para eventual novo centro de distribuição

## Achados principais

1. **Concentração geográfica desproporcional**: 6 localizações físicas 
   atendem apenas 4 dos 27 estados onde a LH Nautical possui clientes.

2. **85% da receita gerada fora da cobertura física**: R$ 1,14 bi de 
   R$ 1,34 bi total originam-se de estados sem infraestrutura local dedicada.

3. **Campo `channel` não constitui critério decisório**: teste estatístico 
   confirmou que canal de venda (e-commerce/presencial) não reflete padrão 
   operacional real.

4. **Saúde operacional dos CDs existentes**: todos os 3 centros de 
   distribuição apresentam crescimento entre 17% e 20% ao ano, sem necessidade 
   de fechamento ou redução.

5. **Candidatos prioritários identificados**: Ceará, Rio de Janeiro e 
   Paraíba destacam-se pela combinação de receita comprovada, crescimento 
   consistente e ausência de cobertura física.

## Recomendação estratégica

**Questão (c) - CDs atuais**: Manter os 3 centros de distribuição existentes 
sem alterações.

**Questão (d) - Novo CD**: Priorizar abertura de centro de distribuição no 
**Ceará**, seguido por Rio de Janeiro e Paraíba — estados que combinam maior 
volume de demanda comprovado com crescimento mais consistente entre regiões 
atualmente descobertas.

## Abordagem metodológica

Análise descritiva (não preditiva) fundamentada em:
- Receita histórica
- Taxa de crescimento (comparação últimos 12 meses vs. 12 meses anteriores)
- Distribuição geográfica aproximada

Opção metodológica consciente de não empregar modelos preditivos de machine 
learning, dado que a decisão de "onde abrir" envolve localizações sem 
histórico operacional próprio (cenário "cold start", limitação conhecida de 
modelos preditivos).